In [1]:
import h5py
import traceback
import numpy as np
import plotly.graph_objects as go
import matplotlib as mpl
mpl.use('Qt5Agg')  # Use Qt5 backend for GUI to work properly
import matplotlib.pyplot as plt
import scipy.signal as sc
from scipy.optimize import curve_fit
from scipy.ndimage import filters 
import tifffile as tiff
from scipy.signal import find_peaks
import sys
sys.path.insert(0, r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\code")

import plotly.express as px
import os
import csv
import pandas as pd
from roipoly import MultiRoi
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.interpolate import interp1d
import ast
from AnalasysFunction import BurstC,clean_and_parse,motorSp,plotFR,devideTr,plotVolCal,VolToCalIdx,CalSmooth,CorrWindow,ChooseSpk,CalInt,VolToCalIdx,CalAmp,calculate_firing_rate,ChooseCom,LongLIST,SingleSpk,linear_model,quadratic_model,exponential_model,MeanRes,lagOptimaizre
from spike_detection_Qixixn2 import complex_bursts_detection,refine_single_spikes,spike_height_calculation,detect_complex_spikes,refine_all_spikes,plot_trace_with_spikes_pdf,plot_trace_with_spikes_export,plot_trace_with_spikes_html
# Create dictionary
from TRY import LongLIST
from NewinternueronsAnalsys import analyze_block
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr, linregress,ttest_ind
import pickle
from matplotlib.widgets import Slider, Button

In [2]:
def gaussian_smooth(x, sigma_samples: float):
    """Simple Gaussian smoothing (no scipy)."""
    if sigma_samples is None or sigma_samples <= 0:
        return np.asarray(x, float).copy()
    x = np.asarray(x, float)
    radius = int(np.ceil(5 * sigma_samples))
    t = np.arange(-radius, radius + 1)
    k = np.exp(-(t**2) / (2 * sigma_samples**2))
    k /= k.sum()
    return np.convolve(x, k, mode="same")

def spike_train_from_indices(spike_idx, n_samples):
    st = np.zeros(n_samples, dtype=np.int8)
    spike_idx = np.asarray(spike_idx, dtype=int)
    spike_idx = spike_idx[(spike_idx >= 0) & (spike_idx < n_samples)]
    st[spike_idx] = 1
    return st

def baseline_8th_percentile(x):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    return np.percentile(x, 8)

def noise_sd_baseline_mad(x, baseline):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    base = x[x <= baseline]
    if base.size < 20:
        base = x
    med = np.median(base)
    mad = np.median(np.abs(base - med))
    return 1.4826 * mad

def detect_ca_events(dff, fs_ca,thr_k=3.0,min_dur_s=0.10,refractory_s=0.5,smooth_sigma_s=0.0):
    """
    Paper-matching Ca event detection (for OLM):
    - baseline = 8th percentile
    - event threshold = baseline + 3*SD(noise)
    - noise SD estimated robustly from baseline samples
    """
    dff = np.asarray(dff, float)

    # optional smoothing for detection (often 0 is fine at 30 Hz)
    if smooth_sigma_s and smooth_sigma_s > 0:
        dff_used = gaussian_smooth(dff, sigma_samples=smooth_sigma_s * fs_ca)
    else:
        dff_used = dff.copy()

    baseline = baseline_8th_percentile(dff_used)
    noise_sd = noise_sd_baseline_mad(dff_used, baseline)
    thr = baseline + thr_k * noise_sd

    above = dff_used > thr
    idx = np.where(above)[0]
    if idx.size == 0:
        return np.array([], dtype=int), dff_used, thr, baseline, noise_sd

    # contiguous segments
    breaks = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[breaks + 1]]
    ends   = np.r_[idx[breaks], idx[-1]]

    # duration filter
    min_len = int(np.round(min_dur_s * fs_ca))
    keep = (ends - starts + 1) >= max(min_len, 1)
    starts = starts[keep]

    # refractory filter on onsets
    refr = int(np.round(refractory_s * fs_ca))
    onsets = []
    last = -10**12
    for s in starts:
        if s - last >= refr:
            onsets.append(s)
            last = s

    return np.array(onsets, dtype=int), dff_used, thr, baseline, noise_sd
def find_transLen (startIDX,th,calT):
    currTran = calT[startIDX:]
    below = np.where(currTran <= th)[0]  # 1D indices
    if below.size == 0:
        return len(currTran)-1  # or len(currTran)
    
    endIdx = below[0]  # first crossing
    return endIdx

def event_triggered_firing_rate_from_traces(cal_trace,voltage_trace, fs_v,spike_idx_v,ca_onsets, fs_ca,pre_s=1.0, post_s=2.0,fr_bin_s=0.05,fr_smooth_sigma_s=0.10,max_events=None,VolXaX=None,th = 1, CalXax=None):
    """
    Align voltage spikes to Ca event onsets and compute event-triggered firing rate.
    If VolXaX and CalXax are provided, uses timestamp mapping (recommended if any desync).
    Otherwise uses fs-based conversion.
    """
    voltage_trace = np.asarray(voltage_trace)
    n_v = voltage_trace.size
    # cal_trace = np.asarray(cal_trace)
    # n_ca = cal_trace.size
    ca_onsets = np.asarray(ca_onsets, dtype=int)
    

    # Convert Ca onset idx -> voltage idx
    if (VolXaX is not None) and (CalXax is not None):
        VolXaX = np.asarray(VolXaX)
        CalXax = np.asarray(CalXax)
        onset_idx_v = np.array(
            [int(np.argmin(np.abs(VolXaX - CalXax[c]))) for c in ca_onsets],
            dtype=int
        )
        used_time_axis_mapping = True
    else:
        onset_times_s = ca_onsets / fs_ca
        onset_idx_v = np.round(onset_times_s * fs_v).astype(int)
        used_time_axis_mapping = False

    pre_n = int(np.round(pre_s * fs_v))
    post_n = int(np.round(post_s * fs_v))
    pre_n_ca = int(np.round(pre_s * fs_ca))
    post_n_ca = int(np.round(post_s * fs_ca))

    # Filter events too close to edges
    n_ca = len(cal_trace)

    good_v  = (onset_idx_v - pre_n >= 0) & (onset_idx_v + post_n < n_v)
    good_ca = (ca_onsets - pre_n_ca >= 0) & (ca_onsets + post_n_ca < n_ca)

    good = good_v & good_ca

    onset_idx_v = onset_idx_v[good]
    used_ca_onsets = ca_onsets[good]

    # onset_idx_v = onset_idx_v[good]
    # used_ca_onsets = ca_onsets[good]
    win_ca = pre_n_ca + post_n_ca
    caltransend =[]
    
    
   

    if max_events is not None and ca_onsets.size > max_events:
        ca_onsets = ca_onsets[:max_events]
    

    # spike train
    st = spike_train_from_indices(spike_idx_v, n_v).astype(float)

    # FR binning
    bin_n = int(np.round(fr_bin_s * fs_v))
    bin_n = max(bin_n, 1)

    win_n = pre_n + post_n
    n_bins = int(np.floor(win_n / bin_n))
    t_centers = (np.arange(n_bins) + 0.5) * (bin_n / fs_v) - pre_s
    empty_fr_trials = np.zeros((0, n_bins), dtype=float)
    empty_fr_mean = np.zeros((n_bins,), dtype=float)
    empty_fr_sem  = np.zeros((n_bins,), dtype=float)
    empty_cal_trials = np.zeros((0, win_ca), dtype=float)

    if ca_onsets.size == 0:
        return t_centers, empty_fr_trials, empty_fr_mean, empty_fr_sem, np.array([], dtype=int), empty_cal_trials, []

    # ... later after edge filtering:
    if onset_idx_v.size == 0:
        return t_centers, empty_fr_trials, empty_fr_mean, empty_fr_sem, np.array([], dtype=int), empty_cal_trials, []
    fr_trials = np.zeros((onset_idx_v.size, n_bins), dtype=float)
    cal_trials = np.zeros((used_ca_onsets.size,pre_n_ca+post_n_ca))
    for i, o in enumerate(onset_idx_v):
        seg = st[o - pre_n : o + post_n]
        seg = seg[: n_bins * bin_n]
        seg_b = seg.reshape(n_bins, bin_n).sum(axis=1)
        fr_trials[i] = seg_b / (bin_n / fs_v)
    for i, o in enumerate(used_ca_onsets):
        seg = cal_trace[o - pre_n_ca : o + post_n_ca]
        cal_trials[i] = seg
        calE = find_transLen(o,th,cal_trace)
        caltransend.append(calE)
    # smooth across time bins
    if fr_smooth_sigma_s and fr_smooth_sigma_s > 0:
        sigma_bins = fr_smooth_sigma_s / fr_bin_s
        for i in range(fr_trials.shape[0]):
            fr_trials[i] = gaussian_smooth(fr_trials[i], sigma_samples=sigma_bins)

    fr_mean = fr_trials.mean(axis=0)
    fr_sem = (fr_trials.std(axis=0, ddof=1) / np.sqrt(fr_trials.shape[0])
              if fr_trials.shape[0] > 1 else np.zeros_like(fr_mean))

    return t_centers, fr_trials, fr_mean, fr_sem, used_ca_onsets,cal_trials,caltransend

def optionB_ca_event_windows_compare_fr(ca_dff, fs_ca,voltage_trace, fs_v,spike_idx_v,CalXax=None, VolXaX=None,thr_k=3.0,ca_smooth_sigma_s=0.0,min_event_dur_s=0.10,refractory_s=0.5,
    pre_s=1.0,post_s=2.0,fr_bin_s=0.05,fr_smooth_sigma_s=0.10,max_events=None,save_path=None,name=''):
    """
    Option B (Calcium-event-defined windows), paper-matching baseline:
      - baseline = 8th percentile of Ca (ΔF/F) values
      - event threshold = baseline + 3*SD(noise)
    Then computes event-triggered firing rate from voltage spikes around Ca event onsets.

    If save_path is provided, saves output dict as:
        <save_path>/calciumeventsDic.pkl

    Returns:
        calciumeventsDic (dict)
    """
    ca_dff = np.asarray(ca_dff).astype(float)

    # 1) detect Ca events (paper method)
    onsets_ca, ca_used, thr, baseline, noise_sd = detect_ca_events(ca_dff, fs_ca,thr_k=thr_k,min_dur_s=min_event_dur_s,refractory_s=refractory_s,smooth_sigma_s=ca_smooth_sigma_s)

    # 2) event-triggered FR around Ca onsets
    t_centers, fr_trials, fr_mean, fr_sem, used_onsets,cal_trials,CALtransi_end = event_triggered_firing_rate_from_traces(cal_trace = ca_used, voltage_trace=voltage_trace, fs_v=fs_v,spike_idx_v=spike_idx_v,ca_onsets=onsets_ca, fs_ca=fs_ca,pre_s=pre_s, post_s=post_s,fr_bin_s=fr_bin_s, th=thr,
        fr_smooth_sigma_s=fr_smooth_sigma_s,max_events=max_events,VolXaX=VolXaX, CalXax=CalXax)

    calciumeventsDic = {
        "ca_onsets_idx": onsets_ca,
        "ca_onsets_used_idx": used_onsets,
        "ca_dff_for_detection": ca_used,
        "threshold": thr,
        "baseline_8th": baseline,
        "noise_sd": noise_sd,
        "t_fr": t_centers,          # seconds, centered on Ca onset (0 = onset)
        "fr_trials": fr_trials,     # (n_events, n_timebins)
        "cal_trials": cal_trials,
        "cal_end":CALtransi_end,
        "fr_mean": fr_mean,
        "fr_sem": fr_sem,
        "n_events_total": int(onsets_ca.size),
        "n_events_used": int(used_onsets.size),
        "fs_ca": fs_ca,
        "fs_v": fs_v,
        "used_time_axis_mapping": (VolXaX is not None) and (CalXax is not None),
        "params": {
            "thr_k": thr_k,
            "ca_smooth_sigma_s": ca_smooth_sigma_s,
            "min_event_dur_s": min_event_dur_s,
            "refractory_s": refractory_s,
            "pre_s": pre_s,
            "post_s": post_s,
            "fr_bin_s": fr_bin_s,
            "fr_smooth_sigma_s": fr_smooth_sigma_s,
            "max_events": max_events
        }
    }

    # 3) save
    if save_path is not None:
        os.makedirs(save_path, exist_ok=True)
        pkl_path = os.path.join(save_path, f"calciumeventsDic{name}.pkl")
        with open(pkl_path, "wb") as f:
            pickle.dump(calciumeventsDic, f)
        #print(f"Saved calcium events dictionary to:\n{pkl_path}")

    return calciumeventsDic

def _save_plotly_fig(fig, out_basepath_no_ext: str,name = ''):
    html_path = out_basepath_no_ext +"_"+name+ ".html"
    svg_path = out_basepath_no_ext + "_"+name+".svg"
    os.makedirs(os.path.dirname(out_basepath_no_ext) or ".", exist_ok=True)

    fig.write_html(html_path, include_plotlyjs="cdn")
    try:
        fig.write_image(svg_path)  # requires kaleido
    except Exception as e:
        print("⚠️ SVG export failed (likely missing kaleido). Install: pip install -U kaleido")
        print("   Error:", e)

    #print(f"Saved:\n  {html_path}\n  {svg_path}")

def plot_event_triggered_fr_and_ca(calciumeventsDic,out_dir,fname_base="event_triggered_FR_and_Ca",pre_s=None, post_s=None, name = ''):
    t_fr = np.asarray(calciumeventsDic["t_fr"], float)
    fr_trials = calciumeventsDic["fr_trials"]
    cal_trials = calciumeventsDic["cal_trials"]
    fr_mean = np.asarray(calciumeventsDic["fr_mean"], float)

    ca_dff = np.asarray(calciumeventsDic["ca_dff_for_detection"], float)
    fs_ca = float(calciumeventsDic["fs_ca"])
    used_onsets = np.asarray(calciumeventsDic["ca_onsets_used_idx"], int)
    fr_ax = np.linspace(-200, 1000, np.size(fr_trials,1))
    SinSShXaxC = np.arange(-200, 1000, 33)
    

    ca_mean = cal_trials.mean(axis=0)

    # --- Plotly figure with secondary y ---
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    if fr_trials is None or (isinstance(fr_trials, np.ndarray) and fr_trials.size == 0):
        print("⚠️ No usable events for FR plot (fr_trials is None/empty).")
        print("   n_events_total:", calciumeventsDic.get("n_events_total"))
        print("   n_events_used :", calciumeventsDic.get("n_events_used"))
        return None
    if cal_trials is None or (isinstance(cal_trials, np.ndarray) and cal_trials.size == 0):
        print("⚠️ No calcium trials to plot.")
        return None

    # FR single trials
    if fr_trials is not None and np.size(fr_trials) > 0:
        for i in range(fr_trials.shape[0]):
            fig.add_trace(
                go.Scatter(
                    x=fr_ax, y=fr_trials[i],
                    mode="lines",
                    line=dict(width=1),
                    opacity=0.20,
                    name="FR trials" if i == 0 else None,
                    showlegend=(i == 0)
                ),
                secondary_y=False
            )

    # FR mean (bold red)
    fig.add_trace(
        go.Scatter(
            x=fr_ax, y=fr_mean,
            mode="lines",
            line=dict(width=4),
            name="FR mean"
        ),
        secondary_y=False
    )

    # Ca single trials (light blue)
    for i in range(cal_trials.shape[0]):
        fig.add_trace(
            go.Scatter(
                x=SinSShXaxC, y=cal_trials[i],
                mode="lines",
                line=dict(width=1),
                opacity=0.20,
                name="Ca trials" if i == 0 else None,
                showlegend=(i == 0)
            ),
            secondary_y=True
        )

    # Ca mean (bold blue)
    fig.add_trace(
        go.Scatter(
            x=SinSShXaxC, y=ca_mean,
            mode="lines",
            line=dict(width=4),
            name="Ca mean"
        ),
        secondary_y=True
    )

    for tr in fig.data:
        if tr.name in (None, "FR trials"):
            # FR trial traces (legend only for first)
            if tr.showlegend or tr.name is None:
                tr.line.color = "rgba(255,0,0,0.25)"
            else:
                tr.line.color = "rgba(255,0,0,0.25)"
        if tr.name == "FR mean":
            tr.line.color = "rgba(255,0,0,1.0)"
        if tr.name in (None, "Ca trials"):
            # Ca trial traces (legend only for first)
            # Note: some traces have name None; we won't overwrite FR ones by checking yaxis
            pass

    # Fix: set Ca traces by their axis (secondary y uses yaxis2)
    for tr in fig.data:
        if getattr(tr, "yaxis", "y") == "y2":
            if tr.name == "Ca mean":
                tr.line.color = "rgba(0,0,255,1.0)"
            else:
                tr.line.color = "rgba(0,0,255,0.25)"

    fig.update_layout(
        title="Event-triggered firing rate and calcium (aligned to Ca onset)",
        xaxis_title="Time (s) relative to Ca onset",
        yaxis_title="Firing rate (Hz)",
        yaxis2_title="ΔF/F",
        template="simple_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        width=1100,
        height=550
    )
    fig.add_vline(x=0, line_width=2, line_dash="dash", line_color="black")

    out_base = os.path.join(out_dir, fname_base)
    _save_plotly_fig(fig, out_base,name)
    return fig

def plot_whole_trace_voltage_calcium_with_events(voltage_trace, VolXaX,ca_dff, CalXax,spike_idx_v,calciumeventsDic=None,out_dir=".",fname_base="whole_trace_Voltage_Ca_events",threshold=None,fs_ca=30,min_event_dur_s=0.10,refractory_s=0.5,name = ''):
    """
    Plots whole voltage (red) and calcium (blue) with:
      - spikes as black dotted markers
      - each calcium transient marked by a CYAN rectangle (onset->end)
    Shared x axis, secondary y.

    If calciumeventsDic is provided, uses:
      - calciumeventsDic['threshold']
      - calciumeventsDic['fs_ca']
      - calciumeventsDic['params'] (min_event_dur_s, refractory_s) if present
      - calciumeventsDic['ca_dff_for_detection'] as ca signal for event intervals
    """
    voltage_trace = np.asarray(voltage_trace, float)
    VolXaX = np.asarray(VolXaX, float)
    ca_dff = np.asarray(ca_dff, float)
    CalXax = np.asarray(CalXax, float)
    spike_idx_v = np.asarray(spike_idx_v, int)

    # Decide detection inputs for cyan boxes
    if calciumeventsDic is not None:
        cal_ev = calciumeventsDic["ca_onsets_used_idx"]
        cal_en = calciumeventsDic["cal_end"]
    else:
        if threshold is None:
            raise ValueError("Provide either calciumeventsDic or an explicit `threshold` for Ca event boxes.")
        

    

    # Map spike indices to times on VolXaX
    spike_idx_v = spike_idx_v[(spike_idx_v >= 0) & (spike_idx_v < VolXaX.size)]
    spike_t = VolXaX[spike_idx_v]
    spike_y = voltage_trace[spike_idx_v]

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Voltage trace (red)
    fig.add_trace(
        go.Scatter(
            x=VolXaX, y=voltage_trace,
            mode="lines",
            line=dict(width=1.5, color="rgba(255,0,0,1.0)"),
            name="Voltage"
        ),
        secondary_y=False
    )

    # Spikes as black dotted markers
    fig.add_trace(
        go.Scatter(
            x=spike_t, y=spike_y,
            mode="markers",
            marker=dict(size=6, color="black", symbol="circle"),
            name="Spikes"
        ),
        secondary_y=False
    )

    # Calcium trace (blue)
    fig.add_trace(
        go.Scatter(
            x=CalXax, y=ca_dff,
            mode="lines",
            line=dict(width=1.5, color="rgba(0,0,255,1.0)"),
            name="Calcium (ΔF/F)"
        ),
        secondary_y=True
    )

    # Cyan boxes for calcium transients (onset->end)
    # shapes are in x (time) coordinates; y spans entire plot by using yref='paper'
    shapes = []
    cal_ev = np.asarray(cal_ev, dtype=int)
    cal_en = np.asarray(cal_en, dtype=int)

    for onset, end_off in zip(cal_ev, cal_en):
        if onset < 0 or onset >= CalXax.size:
            continue

        end_abs = onset + end_off  # convert offset -> absolute index
        end_abs = int(np.clip(end_abs, 0, CalXax.size - 1))

        x0 = float(CalXax[onset])
        x1 = float(CalXax[end_abs])
        shapes.append(
                dict(
                    type="rect",
                    xref="x",
                    yref="paper",
                    x0=x0,
                    x1=x1,
                    y0=0,
                    y1=max(ca_dff),
                    line=dict(
                        color="rgba(0,255,255,0.4)",  # cyan
                        width=2,
                        dash="solid"                  # or "dash" if you prefer
                    )
                )
            )

    fig.update_layout(shapes=shapes)

    fig.update_layout(
        title="Whole-trace voltage and calcium with spikes and Ca event boxes",
        xaxis_title="Time (s)",
        yaxis_title="Voltage ΔF/F",
        yaxis2_title="cal ΔF/F",
        template="simple_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        width=1200,
        height=600
    )

    out_base = os.path.join(out_dir, fname_base)
    _save_plotly_fig(fig, out_base,name)
    return fig

def plot_fr_basic(voltage_trace,VolXaX,spike_idx_v,ca_smooth,CalXax,fr_on_ca,fr_change_idx=None,title="Voltage, spikes, calcium, and firing rate",fname_base="base_FR_full_trace",pearson_r=None,
    pearson_p=None,
    r_values_for_box=None,path = '',name = ''):
    voltage_trace = np.asarray(voltage_trace, float)
    VolXaX = np.asarray(VolXaX, float)
    ca_smooth = np.asarray(ca_smooth, float)
    CalXax = np.asarray(CalXax, float)
    fr_on_ca = np.asarray(fr_on_ca, float)
    spike_idx_v = np.asarray(spike_idx_v, dtype=int)

    # --- spike times & values from voltage ---
    valid = (spike_idx_v >= 0) & (spike_idx_v < voltage_trace.size)
    spike_idx_v = spike_idx_v[valid]
    spike_t = VolXaX[spike_idx_v]
    spike_y = voltage_trace[spike_idx_v]

    # --- Create figure ---
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.05,
        specs=[[{"secondary_y": True}],
           [{"secondary_y": True}]]
    )

    fig.add_trace(
        go.Scatter(
            x=VolXaX, y=voltage_trace,
            mode="lines",
            line=dict(width=1.5, color="rgba(255,0,0,1.0)"),
            name="Voltage"
        ),
        row=1, col=1, secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=spike_t, y=spike_y,
            mode="markers",
            marker=dict(size=6, color="black", symbol="circle"),
            name="Spikes"
        ),
        row=1, col=1, secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=CalXax, y=ca_smooth,
            mode="lines",
            line=dict(width=1.5, color="rgba(0,0,255,1.0)"),
            name="Calcium (smoothed)"
        ),
        row=1, col=1, secondary_y=True
    )

    fig.add_trace(
        go.Scatter(
            x=CalXax, y=fr_on_ca,
            mode="lines",
            line=dict(width=2, color="rgba(255,0,0,1.0)"),
            name="Firing rate (Hz)"
        ),
        row=2, col=1, secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=CalXax, y=ca_smooth,
            mode="lines",
            line=dict(width=1.5, color="rgba(0,0,255,1.0)"),
            name="Calcium (smoothed)",
            showlegend=False
        ),
        row=2, col=1, secondary_y=True
    )
    for idx in fr_change_idx:
            if idx < 0 or idx >= CalXax.size:
                continue
            fig.add_vline(
                x=float(CalXax[idx]),
                line_width=2,
                line_dash="dash",
                line_color="black",
                row=2,
                col=1
            )

    fig.update_layout(
        title=title,
        template="simple_white",
        width=1200,
        height=750,
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="left", x=0
        ),
    )

    fig.update_yaxes(title_text="Voltage (mV)", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="ΔF/F", row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text="FR (Hz)", row=2, col=1, secondary_y=False)
    fig.update_yaxes(title_text="ΔF/F",   row=2, col=1, secondary_y=True)
    fig.update_xaxes(title_text="Time (s)", row=2, col=1)

    if pearson_r is not None and pearson_p is not None:
        fig.add_annotation(
            xref="paper", yref="paper",
            x=0.75, y=0.15,
            text=f"<b>Pearson</b><br>r = {pearson_r:.3f}<br>p = {pearson_p:.2e}",
            showarrow=False,
            align="right",
            font=dict(size=13),
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="black",
            borderwidth=1
        )

    out_base = os.path.join(path, fname_base)
    _save_plotly_fig(fig, out_base,name)
    return fig


In [3]:


def pick_peak_windows_in_runs(corr_list, win_t, thr=0.5):
    """
    From windows where corr > thr, group sequential indices into runs,
    and keep only the single best (max corr) window in each run.

    Returns:
      keep_idx: np.ndarray of indices (one per run)
    """
    corr = np.asarray(corr_list, dtype=float)
    win_t = np.asarray(win_t, dtype=float)  # (n,2)
    good = np.isfinite(corr) & (corr > thr)
    idx = np.where(good)[0]
    if idx.size == 0:
        return np.array([], dtype=int)

    keep = []
    run_start = 0
    # split into runs where indices are consecutive
    for k in range(1, idx.size):
        if idx[k] != idx[k - 1] + 1:
            run = idx[run_start:k]
            best = run[np.nanargmax(corr[run])]
            keep.append(int(best))
            run_start = k

    # last run
    run = idx[run_start:]
    best = run[np.nanargmax(corr[run])]
    keep.append(int(best))

    return np.asarray(keep, dtype=int)


def add_highcorr_window_markers(fig, epoch_corr, thr=0.7, *,
                               shade_rgba="rgba(0,200,0,0.12)",
                               line_rgba="rgba(0,150,0,0.9)",
                               line_width=2,
                               annotate=True):
    """
    Adds markings for high-correlation windows to an existing plotly figure.
    Keeps only the max-r window per consecutive run.
    Marks on BOTH rows (row 1 and row 2), col 1.
    """
    win_t = np.asarray(epoch_corr["win_t"], dtype=float)
    corr = np.asarray(epoch_corr["corr_list"], dtype=float)

    keep_idx = pick_peak_windows_in_runs(corr, win_t, thr=thr)
    for i in keep_idx:
        start_t = float(win_t[i, 0])
        end_t   = float(win_t[i, 1])
        r_val   = float(corr[i])

        # shaded region
        fig.add_vrect(
            x0=start_t, x1=end_t,
            fillcolor=shade_rgba,
            line_width=0,
            row=1, col=1
        )
        fig.add_vrect(
            x0=start_t, x1=end_t,
            fillcolor=shade_rgba,
            line_width=0,
            row=2, col=1
        )

        # boundary lines
        for x in (start_t, end_t):
            fig.add_vline(
                x=x,
                line_width=line_width,
                line_dash="solid",
                line_color=line_rgba,
                row=1, col=1
            )
            fig.add_vline(
                x=x,
                line_width=line_width,
                line_dash="solid",
                line_color=line_rgba,
                row=2, col=1
            )

        if annotate:
            # Put annotation in row2 area; xref is global axis though, so use paper coords for y
            fig.add_annotation(
                x=(start_t + end_t) / 2.0,
                yref="paper",
                y=0.02,  # near bottom
                text=f"r={r_val:.2f}",
                showarrow=False,
                font=dict(size=11),
                bgcolor="rgba(255,255,255,0.7)",
                bordercolor="rgba(0,150,0,0.9)",
                borderwidth=1
            )

    return fig


In [4]:
def baseline_mode_hist(x, n_bins=200, return_details=False):
    """
    Baseline = mode of the histogram (most frequent value).

    Args:
        x: 1D array (e.g., dF/F or fluorescence)
        n_bins: number of histogram bins
        return_details: if True, also return (hist, bin_edges, mode_bin_index)

    Returns:
        baseline_mode (float)  [and optionally details]
    """
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        raise ValueError("Input is empty after removing NaNs.")

    hist, edges = np.histogram(x, bins=n_bins)
    idx = int(np.argmax(hist))
    baseline = 0.5 * (edges[idx] + edges[idx + 1])  # center of the most populated bin

    if return_details:
        return baseline, hist, edges, idx
    return baseline
def find_cal_trans(calT,calSmooth,spikeID,volT,calAX,volAx,thK,winB,winAf,SR_cal,SR_vol):
    smoothenCal = gaussian_smooth(calT,calSmooth)
    calBase = baseline_mode_hist(smoothenCal)
    calSTD = noise_sd_baseline_mad(smoothenCal,calBase)
    susTans = np.argwhere(smoothenCal > 0)
def robust_mad_sigma(x):
    """Robust sigma estimate from MAD."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size < 10:
        return np.nan
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return 1.4826 * mad
def fr_corr_and_change_points_scipy(cal_trace, fs_ca,spike_idx_v, fs_v,fr_bin_s=0.2,fr_smooth_sigma_s=0.3,ca_smooth_sigma_s=0.0,change_method="mad",   # "mad" or "zscore"
                                    change_k=3.0,min_separation_s=0.5,):
    """
    Builds FR(t), resamples it to Ca time base, computes Pearson r (SciPy) between FR and Ca,
    and finds FR change points where |dFR| exceeds threshold.

    Returns dict with:
      t_ca, ca_used, fr_on_ca,
      pearson_r, pearson_p,
      fr_change_idx, fr_change_t, fr_change_sign,
      dfr, thr_dfr
    """
    cal_trace = np.asarray(cal_trace, float)
    n_ca = cal_trace.size
    t_ca = np.arange(n_ca) / float(fs_ca)
    fr_traces = []
    cal_traces = []
    # Ca smoothing (optional)
    ca_used = gaussian_smooth(cal_trace, ca_smooth_sigma_s * fs_ca) if ca_smooth_sigma_s > 0 else cal_trace.copy()
    # Build FR on bins up to Ca duration
    spike_idx_v = np.asarray(spike_idx_v, dtype=int)
    if spike_idx_v.size == 0:
        fr_on_ca = np.zeros(n_ca, float)
    else:
        spike_t = spike_idx_v / float(fs_v)
        duration_s = t_ca[-1] if n_ca > 1 else 0.0

        bin_w = max(float(fr_bin_s), 1e-6)
        edges = np.arange(0, duration_s + bin_w, bin_w)
        if edges.size < 2:
            edges = np.array([0.0, bin_w])

        counts, _ = np.histogram(spike_t, bins=edges)
        fr = counts / bin_w
        t_fr = (edges[:-1] + edges[1:]) / 2.0

        # FR smoothing (optional)
        if fr_smooth_sigma_s and fr_smooth_sigma_s > 0:
            fr = gaussian_smooth(fr, sigma_samples=(fr_smooth_sigma_s / bin_w))

        # interpolate to Ca axis
        fr_on_ca = np.interp(t_ca, t_fr, fr, left=fr[0], right=fr[-1])

    # --- Pearson correlation (SciPy default pearsonr) ---
    m = np.isfinite(fr_on_ca) & np.isfinite(ca_used)
    if m.sum() < 5:
        pearson_r, pearson_p = np.nan, np.nan
    else:
        x = fr_on_ca[m]
        y = ca_used[m]
        # pearsonr errors if constant arrays
        if np.std(x) == 0 or np.std(y) == 0:
            pearson_r, pearson_p = np.nan, np.nan
        else:
            pearson_r, pearson_p = pearsonr(x, y)
    
    if np.isfinite(pearson_r):
        r_obs_perm, p_perm, r_null = permutation_test_corr_circular(
            x=fr_on_ca,
            y=ca_used,
            n_perm=2000,
            min_shift=15,   # ≥0.5 s shift to destroy alignment
            random_state=0
        )
    else:
        p_perm = np.nan
        r_null = np.array([])

    # --- Change points on FR using dFR ---
    dfr = np.diff(fr_on_ca, prepend=fr_on_ca[0])

    if change_method.lower() == "zscore":
        sigma = np.nanstd(dfr)
        thr = change_k * sigma
    else:  # "mad"
        sigma = robust_mad_sigma(dfr)
        thr = change_k * sigma

    if (thr is None) or (not np.isfinite(thr)) or thr <= 0:
        fr_change_idx = np.array([], dtype=int)
        fr_change_sign = np.array([], dtype=int)
    else:
        cand = np.where(np.abs(dfr) >= thr)[0]
        if cand.size == 0:
            fr_change_idx = np.array([], dtype=int)
            fr_change_sign = np.array([], dtype=int)
        else:
            min_sep = max(int(round(min_separation_s * fs_ca)), 1)

            kept = [int(cand[0])]
            s = max(0,cand[0]-15)
            e = min(len(fr_on_ca)-1,cand[0]+16)
            fr_traces.append(fr_on_ca[s:e])
            cal_traces.append(ca_used[s:e])
            for x in cand[1:]:
                x = int(x)
                if x - kept[-1] >= min_sep:
                    kept.append(x)
                    s = max(0,x-15)
                    e = min(len(fr_on_ca)-1,x+16)
                    fr_traces.append(fr_on_ca[s:e])
                    cal_traces.append(ca_used[s:e])
                else:
                    # keep stronger one within cluster
                    if abs(dfr[x]) > abs(dfr[kept[-1]]):
                        kept[-1] = x
                        s = max(0,x-15)
                        e = min(len(fr_on_ca)-1,x+16)
                        fr_traces[-1]=(fr_on_ca[s:e])
                        cal_traces[-1]=(ca_used[s:e])

            fr_change_idx = np.asarray(kept, dtype=int)
            fr_change_sign = np.sign(dfr[fr_change_idx]).astype(int)
            fr_change_sign[fr_change_sign == 0] = 1

    fr_change_t = t_ca[fr_change_idx] if fr_change_idx.size else np.array([], float)
    
    return {
        "t_ca": t_ca,
        "ca_used": ca_used,
        "fr_on_ca": fr_on_ca,
        "pearson_r": float(pearson_r) if np.isfinite(pearson_r) else np.nan,
        "pearson_p": float(pearson_p) if np.isfinite(pearson_p) else np.nan,
        "pearson_p_perm": float(p_perm) if np.isfinite(p_perm) else np.nan,
        "pearson_r_null": r_null,
        "fr_change_idx": fr_change_idx,
        "fr_change_t": fr_change_t,
        "fr_change_sign": fr_change_sign,
        "dfr": dfr,
        "thr_dfr": thr,
        "diff_FR_T":fr_traces,
        "diff_cal_T":cal_traces,
        "params": dict(
            fr_bin_s=fr_bin_s,
            fr_smooth_sigma_s=fr_smooth_sigma_s,
            ca_smooth_sigma_s=ca_smooth_sigma_s,
            change_method=change_method,
            change_k=change_k,
            min_separation_s=min_separation_s,
        )
    }

def lag_scan_pearson(
    x, y,
    fs,
    max_lag_s=2.0,
    lag_step_s=None,
    min_overlap_s=1.0,
    mode="max_abs",   # "max_abs" or "max_pos" or "max_neg"
):
    """
    Scan Pearson correlation across time lags between x and y.

    Parameters
    ----------
    x, y : 1D arrays (same sampling rate fs). Typically:
           x = fr_on_ca
           y = ca_smooth
    fs : float
         sampling rate of x/y (for Ca timebase use fs_ca)
    max_lag_s : float
         scan lags in [-max_lag_s, +max_lag_s]
    lag_step_s : float or None
         lag step in seconds; if None -> 1 sample steps
    min_overlap_s : float
         minimum required overlap duration to compute r
    mode : str
         "max_abs" -> maximize |r|
         "max_pos" -> maximize r
         "max_neg" -> minimize r (most negative)

    Returns
    -------
    out : dict with
        best_r, best_p, best_lag_s, best_lag_samples,
        lags_s, r_curve, p_curve
    """
    x = np.asarray(x, float).ravel()
    y = np.asarray(y, float).ravel()

    n = min(len(x), len(y))
    x = x[:n]
    y = y[:n]

    # remove NaNs pairwise later; still keep indices aligned
    max_lag_samp = int(round(max_lag_s * fs))
    step_samp = 1 if (lag_step_s is None) else max(int(round(lag_step_s * fs)), 1)

    lags = np.arange(-max_lag_samp, max_lag_samp + 1, step_samp, dtype=int)

    min_overlap = max(int(round(min_overlap_s * fs)), 3)

    r_curve = np.full(lags.shape, np.nan, float)
    p_curve = np.full(lags.shape, np.nan, float)

    for i, lag in enumerate(lags):
        if lag >= 0:
            # x leads y: compare x[t] with y[t+lag]
            x_seg = x[: n - lag]
            y_seg = y[lag:]
        else:
            # y leads x: compare x[t-lag] with y[t]
            k = -lag
            x_seg = x[k:]
            y_seg = y[: n - k]

        # pairwise finite mask
        m = np.isfinite(x_seg) & np.isfinite(y_seg)
        if m.sum() < min_overlap:
            continue

        r, p = pearsonr(x_seg[m], y_seg[m])
        r_curve[i] = r
        p_curve[i] = p

    # pick best lag
    valid = np.isfinite(r_curve)
    if not np.any(valid):
        return {
            "best_r": np.nan, "best_p": np.nan,
            "best_lag_s": np.nan, "best_lag_samples": None,
            "lags_s": lags / fs, "r_curve": r_curve, "p_curve": p_curve
        }

    if mode == "max_abs":
        idx = np.nanargmax(np.abs(r_curve))
    elif mode == "max_pos":
        idx = np.nanargmax(r_curve)
    elif mode == "max_neg":
        idx = np.nanargmin(r_curve)
    else:
        raise ValueError("mode must be one of: 'max_abs', 'max_pos', 'max_neg'")

    best_lag = int(lags[idx])
    return {
        "best_r": float(r_curve[idx]),
        "best_p": float(p_curve[idx]),
        "best_lag_s": float(best_lag / fs),
        "best_lag_samples": best_lag,
        "lags_s": lags / fs,
        "r_curve": r_curve,
        "p_curve": p_curve,
    }

def plot_perm_histogram_plotly(r_null, r_obs, r_opt, title,r_crit):
    """
    Histogram of permutation null correlations + vertical lines for:
      - observed correlation (zero-lag)
      - optimal lag correlation
    """
    r_null = np.asarray(r_null, float)
    r_null = r_null[np.isfinite(r_null)]

    fig = go.Figure()

    fig.add_trace(go.Histogram(
        x=r_null,
        nbinsx=60,
        name="Permutation null r",
        opacity=0.75
    ))

    if np.isfinite(r_obs):
        fig.add_vline(
            x=r_obs,
            line_width=3,
            line_dash="solid",
            annotation_text="Observed r (0 lag)",
            annotation_position="top"
        )

    if np.isfinite(r_opt):
        fig.add_vline(
            x=r_opt,
            line_width=3,
            line_dash="dash",
            annotation_text="Best-lag r",
            annotation_position="bottom"
        )
    
    fig.add_vline(
        x= r_crit,
        line_dash="dot",
        line_color="black",
        annotation_text="α=0.05 threshold",
    )

    fig.add_vline(
        x=-r_crit,
        line_dash="dot",
        line_color="black",
    )

    fig.update_layout(
        title=title,
        xaxis_title="Pearson r",
        yaxis_title="Count",
        bargap=0.02,
        width=900,
        height=550
    )
    return fig

def save_plotly_svg_html(fig, svg_path, html_path):
    """
    Saves Plotly as SVG + HTML.
    Requires: pip install -U kaleido
    """
    fig.write_image(svg_path, format="svg")
    fig.write_html(html_path, include_plotlyjs="cdn")


In [5]:


def baseline_mode_hist(x, n_bins=200):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    hist, edges = np.histogram(x, bins=n_bins)
    idx = int(np.argmax(hist))
    return 0.5 * (edges[idx] + edges[idx + 1])

def noise_sd_baseline_mad(x, baseline):
    # robust noise estimate around baseline
    x = np.asarray(x, float)
    r = x - baseline
    mad = np.median(np.abs(r - np.median(r)))
    return 1.4826 * mad if mad > 0 else np.std(r)

def find_calcium_transients(
    ca, fs_ca,
    smooth_sigma_s=0.25,
    k=3.0,
    min_dur_s=0.25,
    merge_gap_s=0.15,
    baseline_bins=200
):
    """
    Returns:
      transients: list of dicts with keys:
        start_idx, end_idx (inclusive), peak_idx, baseline, thr
    """
    ca = np.asarray(ca, float)
    sigma_samples = smooth_sigma_s * fs_ca
    ca_s = gaussian_smooth_1d(ca, sigma_samples)

    baseline = baseline_mode_hist(ca_s, n_bins=baseline_bins)
    noise_sd = noise_sd_baseline_mad(ca_s, baseline)
    thr = baseline + k * noise_sd

    above = ca_s > thr
    idx = np.flatnonzero(above)
    if idx.size == 0:
        return [], ca_s, baseline, thr

    # contiguous segments
    cuts = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[cuts + 1]]
    ends   = np.r_[idx[cuts], idx[-1]]

    # merge segments separated by short gaps
    max_gap = int(round(merge_gap_s * fs_ca))
    merged_s, merged_e = [int(starts[0])], [int(ends[0])]
    for s, e in zip(starts[1:], ends[1:]):
        s, e = int(s), int(e)
        if s - merged_e[-1] - 1 <= max_gap:
            merged_e[-1] = e
        else:
            merged_s.append(s); merged_e.append(e)

    # enforce min duration
    min_len = int(round(min_dur_s * fs_ca))
    transients = []
    for s, e in zip(merged_s, merged_e):
        if (e - s + 1) < min_len:
            continue
        seg = ca_s[s:e+1]
        peak_idx = s + int(np.nanargmax(seg))
        transients.append(
            dict(start_idx=s, end_idx=e, peak_idx=peak_idx, baseline=baseline, thr=thr)
        )

    return transients, ca_s, baseline, thr

def fr_corr_and_windowed(
    cal_trace, fs_ca,
    spike_idx_v, fs_v,
    window_dur_s=10.0,
    fr_bin_s=0.2,
    fr_smooth_sigma_s=0.3,
    ca_smooth_sigma_s=0.0,
):
    """
    Divide cal_trace and spike_idx_v into non-overlapping windows (default 10 sec),
    compute Pearson r for each window independently.

    Returns dict with:
      window_r_list, window_p_list, window_times (centers),
      n_windows, params
    """
    cal_trace = np.asarray(cal_trace, float)
    n_ca = cal_trace.size
    t_ca = np.arange(n_ca) / float(fs_ca)
    duration_s = t_ca[-1]

    # Define non-overlapping windows
    window_samples = int(np.round(window_dur_s * fs_ca))
    n_windows = int(np.floor(n_ca / window_samples))

    if n_windows == 0:
        return {
            "window_r_list": np.array([], float),
            "window_p_list": np.array([], float),
            "window_times": np.array([], float),
            "n_windows": 0,
            "params": dict(window_dur_s=window_dur_s, fs_ca=fs_ca, fs_v=fs_v)
        }

    window_r_list = []
    window_p_list = []
    window_times = []

    spike_idx_v = np.asarray(spike_idx_v, dtype=int)

    for w in range(n_windows):
        start_ca = w * window_samples
        end_ca = start_ca + window_samples
        
        # Clamp to valid range
        end_ca = min(end_ca, n_ca)

        # Extract window
        cal_win = cal_trace[start_ca:end_ca]
        if cal_win.size == 0:
            continue

        # Smooth Ca (optional)
        if ca_smooth_sigma_s and ca_smooth_sigma_s > 0:
            ca_win = gaussian_smooth(cal_win, ca_smooth_sigma_s * fs_ca)
        else:
            ca_win = cal_win.copy()

        # Time axis for this window
        t_win = np.arange(start_ca, end_ca) / float(fs_ca)
        n_win = len(t_win)

        # Convert window time to voltage indices
        t_start_s = t_win[0]
        t_end_s = t_win[-1]
        idx_start_v = int(t_start_s * fs_v)
        idx_end_v = int(t_end_s * fs_v)

        # Filter spikes in this window
        spikes_in_win = spike_idx_v[
            (spike_idx_v >= idx_start_v) & (spike_idx_v < idx_end_v)
        ]

        # Compute FR on Ca timebase for this window
        if spikes_in_win.size == 0:
            fr_win = np.zeros(n_win, float)
        else:
            # Spike times in seconds
            spike_t = spikes_in_win / float(fs_v)

            # Bin spikes
            bin_w = max(float(fr_bin_s), 1e-6)
            edges = np.arange(t_start_s, t_end_s + bin_w, bin_w)
            if edges.size < 2:
                edges = np.array([t_start_s, t_end_s + bin_w])

            counts, _ = np.histogram(spike_t, bins=edges)
            fr = counts / bin_w
            t_fr = (edges[:-1] + edges[1:]) / 2.0

            # Smooth FR
            if fr_smooth_sigma_s and fr_smooth_sigma_s > 0:
                fr = gaussian_smooth(fr, sigma_samples=(fr_smooth_sigma_s / bin_w))

            # Interpolate to Ca axis
            fr_win = np.interp(t_win, t_fr, fr, left=fr[0], right=fr[-1])

        # Compute Pearson for this window
        m = np.isfinite(fr_win) & np.isfinite(ca_win)
        if m.sum() < 5:
            r, p = np.nan, np.nan
        else:
            x = fr_win[m]
            y = ca_win[m]
            if np.std(x) == 0 or np.std(y) == 0:
                r, p = np.nan, np.nan
            else:
                r, p = pearsonr(x, y)

        window_r_list.append(float(r) if np.isfinite(r) else np.nan)
        window_p_list.append(float(p) if np.isfinite(p) else np.nan)
        window_times.append((t_start_s + t_end_s) / 2.0)  # center of window

    return {
        "window_r_list": np.asarray(window_r_list, float),
        "window_p_list": np.asarray(window_p_list, float),
        "window_times": np.asarray(window_times, float),
        "n_windows": len(window_r_list),
        "params": dict(
            window_dur_s=window_dur_s,
            fr_bin_s=fr_bin_s,
            fr_smooth_sigma_s=fr_smooth_sigma_s,
            ca_smooth_sigma_s=ca_smooth_sigma_s,
            fs_ca=fs_ca,
            fs_v=fs_v,
        )
    }

In [6]:
def plot_ca_vol_with_transients_purple(
    ca, vol,
    fs_ca=30, fs_vol=500,
    ca_t=None, vol_t=None,
    transients=None,
    title="Calcium + Voltage (transients highlighted)"
):
    ca = np.asarray(ca).flatten()
    vol = np.asarray(vol).flatten()

    if ca_t is None:
        ca_t = np.arange(ca.size) / fs_ca
    if vol_t is None:
        vol_t = np.arange(vol.size) / fs_vol

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=vol_t, y=vol, name="Voltage", line=dict(width=1)),
                  secondary_y=False)
    fig.add_trace(go.Scatter(x=ca_t, y=ca, name="Calcium", line=dict(width=2, color="black")),
                  secondary_y=True)

    # add purple overlays for each transient segment (on calcium trace)
    if transients is not None:
        for j, tr in enumerate(transients, start=1):
            s, e = tr["start_idx"], tr["end_idx"]
            fig.add_trace(
                go.Scatter(
                    x=ca_t[s:e+1], y=ca[s:e+1],
                    mode="lines",
                    name=f"Ca transient {j}",
                    line=dict(width=4, color="purple"),
                    showlegend=(j == 1)  # keep legend clean
                ),
                secondary_y=True
            )

    fig.update_layout(title=title, xaxis_title="Time (s)")
    fig.update_yaxes(title_text="Voltage", secondary_y=False)
    fig.update_yaxes(title_text="Calcium", secondary_y=True)
    return fig


In [7]:
DB = pd.read_csv(r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\SST_Final.csv')
values = DB['SNR'].tolist()
r = DB
awakePyr = r['Notes']
bsPyr = list(r['brainState'])
pathPyr = list(r['Link'])
AllCalSR = list(r['CALsr'])
subG = list(r['subGroup'])
bsOnlyM = [bsPyr[i] for i, s in enumerate(subG) if s.lower() == "motion"]
onlyMot = [pathPyr[i] for i, s in enumerate(subG) if s.lower() == "motion"]
oNAllCalSR = [AllCalSR[i] for i, s in enumerate(subG) if s.lower() == "motion"]

In [8]:


def permutation_test_corr_circular(
    x, y,
    n_perm=2000,
    min_shift=30,
    random_state=None
):
    """
    Circular-shift permutation test for Pearson correlation.

    Returns
    -------
    r_obs : float
        Observed Pearson correlation (non-permuted)
    p_perm : float
        Two-sided permutation p-value
    r_perm : np.ndarray
        Null distribution of permutation correlation coefficients (length n_perm)
    """
    rng = np.random.default_rng(random_state)

    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    # Edge cases
    if x.size < 10 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan, np.array([])

    r_obs, _ = pearsonr(x, y)

    n = x.size
    min_shift = int(min_shift)

    # Ensure min_shift is valid for short signals
    if min_shift < 1:
        min_shift = 1
    if 2 * min_shift >= n:
        # relax: keep at most n//4 (but >=1)
        min_shift = max(1, n // 4)

    r_perm = np.empty(n_perm, float)

    for i in range(n_perm):
        # shift must be in [min_shift, n-min_shift)
        shift = rng.integers(min_shift, n - min_shift)
        x_shift = np.roll(x, shift)
        r_perm[i], _ = pearsonr(x_shift, y)

    # Two-sided permutation p-value with +1 correction
    p_perm = (np.sum(np.abs(r_perm) >= abs(r_obs)) + 1) / (n_perm + 1)

    return float(r_obs), float(p_perm), r_perm



In [9]:

try:
    from scipy.ndimage import gaussian_filter1d as _gaussian_filter1d
except Exception:  # pragma: no cover
    _gaussian_filter1d = None


def _smooth_gaussian_1d(x, sigma_samples):
    x = np.asarray(x, dtype=float).ravel()
    if sigma_samples is None or sigma_samples <= 0:
        return x
    if _gaussian_filter1d is not None:
        return _gaussian_filter1d(x, sigma=float(sigma_samples), mode="nearest")

    radius = int(np.ceil(4 * float(sigma_samples)))
    if radius < 1:
        return x
    t = np.arange(-radius, radius + 1, dtype=float)
    k = np.exp(-(t**2) / (2.0 * float(sigma_samples) ** 2))
    k /= k.sum()
    return np.convolve(x, k, mode="same")


def windowed_fr_calcium_correlation(
    fr,
    calcium,
    t=None,
    fs=None,
    ws_s=5.0,
    step_s=0.1,
    sigma_s=0.0,
    sigma_fr_s=None,
    sigma_ca_s=None,
):
    """
    Calculate Pearson correlation (FR vs calcium) in sliding time windows.

    Parameters
    ----------
    fr, calcium : array-like (1D)
        Signals with the same length and time base.
    t : array-like (1D), optional
        Time axis in seconds (same length as signals). If None, `fs` must be provided.
    fs : float, optional
        Sampling rate in Hz (used only if `t` is None).
    ws_s : float
        Window size in seconds (e.g. 5.0).
    step_s : float
        Step size in seconds (e.g. 0.1 for 100 ms).
    sigma_s : float
        Default Gaussian smoothing sigma in seconds (applies to both traces unless overridden).
    sigma_fr_s, sigma_ca_s : float, optional
        Gaussian smoothing sigma in seconds for FR / calcium, overrides `sigma_s` when provided.

    Returns
    -------
    corr_list : list[float]
        Pearson r per window (np.nan when undefined).
    win_t : np.ndarray, shape (n_windows, 2)
        Start and end time (seconds) for each window.
    mid_t : list[float]
        Mid-point time (seconds) for each window.
    ws_s : float
        Window size in seconds (echo).
    sigma_used : tuple[float, float]
        (sigma_fr_s, sigma_ca_s) used (seconds).
    """
    fr = np.asarray(fr, dtype=float).ravel()
    calcium = np.asarray(calcium, dtype=float).ravel()
    if fr.shape[0] != calcium.shape[0]:
        raise ValueError(f"fr and calcium must have same length (got {fr.shape[0]} vs {calcium.shape[0]})")
    if fr.size < 2:
        sigma_fr = float(sigma_s if sigma_fr_s is None else sigma_fr_s)
        sigma_ca = float(sigma_s if sigma_ca_s is None else sigma_ca_s)
        return [], np.zeros((0, 2), dtype=float), [], float(ws_s), (sigma_fr, sigma_ca)

    if t is None:
        if fs is None:
            raise ValueError("Provide `t` (seconds) or `fs` (Hz).")
        fs = float(fs)
        if not (fs > 0):
            raise ValueError(f"fs must be > 0 (got {fs})")
        dt = 1.0 / fs
        t0 = 0.0
    else:
        t = np.asarray(t, dtype=float).ravel()
        if t.shape[0] != fr.shape[0]:
            raise ValueError(f"t must have same length as signals (got {t.shape[0]} vs {fr.shape[0]})")
        if t.size < 2:
            sigma_fr = float(sigma_s if sigma_fr_s is None else sigma_fr_s)
            sigma_ca = float(sigma_s if sigma_ca_s is None else sigma_ca_s)
            return [], np.zeros((0, 2), dtype=float), [], float(ws_s), (sigma_fr, sigma_ca)
        dt = float(np.median(np.diff(t)))
        if not (dt > 0):
            raise ValueError("t must be strictly increasing in seconds.")
        t0 = float(t[0])

    ws_s = float(ws_s)
    step_s = float(step_s)
    if not (ws_s > 0):
        raise ValueError(f"ws_s must be > 0 (got {ws_s})")
    if not (step_s > 0):
        raise ValueError(f"step_s must be > 0 (got {step_s})")

    sigma_fr_s = float(sigma_s if sigma_fr_s is None else sigma_fr_s)
    sigma_ca_s = float(sigma_s if sigma_ca_s is None else sigma_ca_s)

    fr = _smooth_gaussian_1d(fr, sigma_samples=sigma_fr_s / dt if sigma_fr_s > 0 else 0.0)
    calcium = _smooth_gaussian_1d(calcium, sigma_samples=sigma_ca_s / dt if sigma_ca_s > 0 else 0.0)

    ws_n = int(round(ws_s / dt))
    step_n = int(round(step_s / dt))
    if ws_n < 2:
        raise ValueError(f"Window too small for dt={dt:.6g}s (ws_s={ws_s} -> ws_n={ws_n} samples)")
    if step_n < 1:
        step_n = 1

    last_start = fr.size - ws_n
    if last_start < 0:
        return [], np.zeros((0, 2), dtype=float), [], float(ws_s), (sigma_fr_s, sigma_ca_s)

    corr_list = []
    window_fr = []
    window_cal=[]
    win_t = []
    mid_t = []
    for start in range(0, last_start + 1, step_n):
        end = start + ws_n
        x = fr[start:end]
        window_fr.append(x)
        y = calcium[start:end]
        window_cal.append(y)

        finite = np.isfinite(x) & np.isfinite(y)
        if finite.sum() < 2:
            r = np.nan
        else:
            xf = x[finite]
            yf = y[finite]
            if np.std(xf) == 0 or np.std(yf) == 0:
                r = np.nan
            else:
                r = float(np.corrcoef(xf, yf)[0, 1])

        start_t = float(t0 + start * dt)
        end_t = float(start_t + ws_n * dt)
        win_t.append((start_t, end_t))
        mid_t.append((start_t + end_t) / 2.0)
        corr_list.append(r)
    result_dict = {
        "corr_list": corr_list,
        "win_t": np.asarray(win_t, dtype=float),
        "mid_t": mid_t,
        "window_size_s": float(ws_s),
        "smoothing_sigmas_s": {
            "sigma_fr_s": sigma_fr_s,
            "sigma_ca_s": sigma_ca_s,
        },
        "window_fr": window_fr,
        "window_cal": window_cal,
    }


    return result_dict


def calculate_fr_diff(eppochs):
    fr = eppochs['window_fr']
    seqDIFFabs = []
    seqDIFF = []
    meanMaxDiff = []
    for f in fr:
       seqDIFFabs.append(float(np.mean(np.abs(np.diff(f)))))
       seqDIFF.append((np.mean((np.diff(f)))))
       MaxFR = max(f)
       MinFR = min(f)
       meanMaxDiff.append(MaxFR-MinFR)
    return seqDIFFabs, seqDIFF, meanMaxDiff


In [10]:


def plot_window_corr_results_plotly_top(
    epoch_corr,
    r_thr=0.7,
    desired_vertical_spacing=0.05,
    per_page=10,
    width=1200,
    row_height=300,
):
    mid_t = np.asarray(epoch_corr["mid_t"], dtype=float)
    corr = np.asarray(epoch_corr["corr_list"], dtype=float)

    window_fr = epoch_corr["window_fr"]
    window_cal = epoch_corr["window_cal"]

    ws_s = float(epoch_corr["window_size_s"])
    sig_fr = float(epoch_corr["smoothing_sigmas_s"]["sigma_fr_s"])
    sig_ca = float(epoch_corr["smoothing_sigmas_s"]["sigma_ca_s"])

    # -------------------------
    # 1) correlation vs time
    # -------------------------
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=mid_t, y=corr, mode="lines", name="Pearson r"))
    fig1.add_trace(go.Scatter(
        x=[float(np.nanmin(mid_t)), float(np.nanmax(mid_t))] if mid_t.size else [0, 1],
        y=[r_thr, r_thr],
        mode="lines",
        name=f"threshold r={r_thr}",
        line=dict(dash="dash")
    ))
    fig1.update_layout(
        title=f"Windowed correlation (ws={ws_s:.2f}s, sig_fr={sig_fr}, sig_ca={sig_ca})",
        xaxis_title="Time (s) (window midpoint)",
        yaxis_title="Pearson r (FR vs calcium)",
        template="plotly_white",
    )

    # ---------------------------------------------------------
    # windows where corr > r_thr
    # ---------------------------------------------------------
    good = np.isfinite(corr) & (corr > r_thr)
    good_idx = np.where(good)[0]
    # ---------------------------------------------------------
    # NEW: fig3 scatter r vs mean-diff(FR) per window
    # ---------------------------------------------------------
    mean_diff_fr = np.full(corr.shape, np.nan, dtype=float)

    for i in range(len(window_fr)):
        fr = np.asarray(window_fr[i], dtype=float).ravel()
        if fr.size < 2:
            continue
        finite = np.isfinite(fr)
        if finite.sum() < 2:
            continue
        frf = fr[finite]
        if frf.size < 2:
            continue
        mean_diff_fr[i] = float(np.mean(np.abs(np.diff(frf))))

    valid_scatter = np.isfinite(corr) & np.isfinite(mean_diff_fr)

    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(
        x=mean_diff_fr[valid_scatter],
        y=corr[valid_scatter],
        mode="markers",
        marker=dict(size=7),
        name="windows"
    ))
    fig3.update_layout(
        title="Correlation vs FR variability per window",
        xaxis_title="Mean |ΔFR| within window (mean absolute first difference)",
        yaxis_title="Pearson r (FR vs calcium)",
        template="plotly_white",
    )
    # -------------------------
    # 2) overlay + mean (fig2)
    # -------------------------
    fig2 = go.Figure()

    if good_idx.size == 0:
        fig2.update_layout(title=f"No windows with r > {r_thr}", template="plotly_white")
        return fig1, fig2,fig3, [],[],[]

    fr_stack = np.vstack([np.asarray(window_fr[i], dtype=float).ravel() for i in good_idx])
    ca_stack = np.vstack([np.asarray(window_cal[i], dtype=float).ravel() for i in good_idx])

    ws_n = fr_stack.shape[1]
    dt = ws_s / ws_n
    xw = np.arange(ws_n) * dt

    dark_red = "rgb(139,0,0)"
    dark_blue = "rgb(0,0,139)"
    light_red = "rgba(255,0,0,0.18)"
    light_blue = "rgba(0,0,255,0.18)"

    for k in range(fr_stack.shape[0]):
        fig2.add_trace(go.Scatter(
            x=xw, y=fr_stack[k],
            mode="lines",
            line=dict(color=light_red, width=1),
            showlegend=False,
            hoverinfo="skip",
            name="FR windows"
        ))
    for k in range(ca_stack.shape[0]):
        fig2.add_trace(go.Scatter(
            x=xw, y=ca_stack[k],
            mode="lines",
            line=dict(color=light_blue, width=1),
            showlegend=False,
            hoverinfo="skip",
            name="Calcium windows"
        ))

    fr_mean = np.nanmean(fr_stack, axis=0)
    ca_mean = np.nanmean(ca_stack, axis=0)

    fig2.add_trace(go.Scatter(
        x=xw, y=fr_mean,
        mode="lines",
        line=dict(color=dark_red, width=3),
        name=f"Mean FR (r>{r_thr})"
    ))
    fig2.add_trace(go.Scatter(
        x=xw, y=ca_mean,
        mode="lines",
        line=dict(color=dark_blue, width=3),
        name=f"Mean Calcium (r>{r_thr})"
    ))

    fig2.update_layout(
        title=f"Windows with r > {r_thr} (n={good_idx.size})",
        xaxis_title="Time within window (s)",
        yaxis_title="Signal value",
        template="plotly_white",
    )

    # -------------------------
    # 3) subplots, PAGED (list of figures) + secondary y-axis per row
    # -------------------------
    win_t = np.asarray(epoch_corr["win_t"], dtype=float)
    fig_subplots = []
    total = int(good_idx.size)

    per_page = int(per_page)
    if per_page < 1:
        per_page = 1

    for page_start in range(0, total, per_page):
        page_idx = good_idx[page_start:page_start + per_page]
        n = int(page_idx.size)

        # Plotly vertical spacing constraint
        if n <= 1:
            vertical_spacing = 0.0
        else:
            max_vs = 1.0 / (n - 1)
            vertical_spacing = min(float(desired_vertical_spacing), max_vs * 0.98)

        titles = [
            f"WinIdx {idx} | r={corr[idx]:.3f} | {win_t[idx,0]:.2f}s–{win_t[idx,1]:.2f}s"
            for idx in page_idx
        ]

        # ✅ Each row gets a secondary y-axis
        fig_subplot = make_subplots(
            rows=n,
            cols=1,
            shared_xaxes=False,
            vertical_spacing=vertical_spacing,
            subplot_titles=titles,
            specs=[[{"secondary_y": True}] for _ in range(n)]
        )

        for row_i, idx in enumerate(page_idx, start=1):
            fr = np.asarray(window_fr[idx], float).ravel()
            cal = np.asarray(window_cal[idx], float).ravel()

            ws_n_local = fr.size
            xw_local = np.arange(ws_n_local) * (ws_s / ws_n_local)

            # FR on left axis
            fig_subplot.add_trace(
                go.Scatter(
                    x=xw_local, y=fr,
                    mode="lines",
                    line=dict(color=dark_red, width=2),
                    name="FR",
                    showlegend=(row_i == 1)
                ),
                row=row_i, col=1, secondary_y=False
            )

            # Calcium on right axis
            fig_subplot.add_trace(
                go.Scatter(
                    x=xw_local, y=cal,
                    mode="lines",
                    line=dict(color=dark_blue, width=2),
                    name="Calcium",
                    showlegend=(row_i == 1)
                ),
                row=row_i, col=1, secondary_y=True
            )

            fig_subplot.update_xaxes(title_text="Time within window (s)", row=row_i, col=1)
            fig_subplot.update_yaxes(title_text="FR (Hz)", row=row_i, col=1, secondary_y=False)
            fig_subplot.update_yaxes(title_text="ΔF/F", row=row_i, col=1, secondary_y=True)

        page_num = page_start // per_page + 1
        fig_subplot.update_layout(
            width=int(width),
            height=int(row_height) * n,
            template="plotly_white",
            title=f"High-corr windows (r>{r_thr}) — page {page_num} ({page_start+1}-{page_start+n} of {total})",
            margin=dict(t=80, b=40, l=70, r=30),
        )

        fig_subplots.append(fig_subplot)

    return fig1, fig2, fig3,fig_subplots,mean_diff_fr,corr




def plot_window_corr_results_plotly_bot(
    epoch_corr,
    r_thr=0.1,
    desired_vertical_spacing=0.05,
    per_page=10,
    width=1200,
    row_height=300,
):
    mid_t = np.asarray(epoch_corr["mid_t"], dtype=float)
    corr = np.asarray(epoch_corr["corr_list"], dtype=float)

    window_fr = epoch_corr["window_fr"]
    window_cal = epoch_corr["window_cal"]

    ws_s = float(epoch_corr["window_size_s"])
    sig_fr = float(epoch_corr["smoothing_sigmas_s"]["sigma_fr_s"])
    sig_ca = float(epoch_corr["smoothing_sigmas_s"]["sigma_ca_s"])

    # -------------------------
    # 1) correlation vs time
    # -------------------------
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=mid_t, y=corr, mode="lines", name="Pearson r"))
    fig1.add_trace(go.Scatter(
        x=[float(np.nanmin(mid_t)), float(np.nanmax(mid_t))] if mid_t.size else [0, 1],
        y=[r_thr, r_thr],
        mode="lines",
        name=f"threshold r={r_thr}",
        line=dict(dash="dash")
    ))
    fig1.update_layout(
        title=f"Windowed correlation (ws={ws_s:.2f}s, sig_fr={sig_fr}, sig_ca={sig_ca})",
        xaxis_title="Time (s) (window midpoint)",
        yaxis_title="Pearson r (FR vs calcium)",
        template="plotly_white",
    )

    # ---------------------------------------------------------
    # windows where corr > r_thr
    # ---------------------------------------------------------
    good = np.isfinite(corr) & (corr < r_thr)
    good_idx = np.where(good)[0]
    # ---------------------------------------------------------
    # NEW: fig3 scatter r vs mean-diff(FR) per window
    # ---------------------------------------------------------
    mean_diff_fr = np.full(corr.shape, np.nan, dtype=float)

    for i in range(len(window_fr)):
        fr = np.asarray(window_fr[i], dtype=float).ravel()
        if fr.size < 2:
            continue
        finite = np.isfinite(fr)
        if finite.sum() < 2:
            continue
        frf = fr[finite]
        if frf.size < 2:
            continue
        mean_diff_fr[i] = float(np.mean(np.abs(np.diff(frf))))

    valid_scatter = np.isfinite(corr) & np.isfinite(mean_diff_fr)

    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(
        x=mean_diff_fr[valid_scatter],
        y=corr[valid_scatter],
        mode="markers",
        marker=dict(size=7),
        name="windows"
    ))
    fig3.update_layout(
        title="Correlation vs FR variability per window",
        xaxis_title="Mean |ΔFR| within window (mean absolute first difference)",
        yaxis_title="Pearson r (FR vs calcium)",
        template="plotly_white",
    )
    # -------------------------
    # 2) overlay + mean (fig2)
    # -------------------------
    fig2 = go.Figure()

    if good_idx.size == 0:
        fig2.update_layout(title=f"No windows with r < {r_thr}", template="plotly_white")
        return fig1, fig2,fig3, []

    fr_stack = np.vstack([np.asarray(window_fr[i], dtype=float).ravel() for i in good_idx])
    ca_stack = np.vstack([np.asarray(window_cal[i], dtype=float).ravel() for i in good_idx])

    ws_n = fr_stack.shape[1]
    dt = ws_s / ws_n
    xw = np.arange(ws_n) * dt

    dark_red = "rgb(139,0,0)"
    dark_blue = "rgb(0,0,139)"
    light_red = "rgba(255,0,0,0.18)"
    light_blue = "rgba(0,0,255,0.18)"

    for k in range(fr_stack.shape[0]):
        fig2.add_trace(go.Scatter(
            x=xw, y=fr_stack[k],
            mode="lines",
            line=dict(color=light_red, width=1),
            showlegend=False,
            hoverinfo="skip",
            name="FR windows"
        ))
    for k in range(ca_stack.shape[0]):
        fig2.add_trace(go.Scatter(
            x=xw, y=ca_stack[k],
            mode="lines",
            line=dict(color=light_blue, width=1),
            showlegend=False,
            hoverinfo="skip",
            name="Calcium windows"
        ))

    fr_mean = np.nanmean(fr_stack, axis=0)
    ca_mean = np.nanmean(ca_stack, axis=0)

    fig2.add_trace(go.Scatter(
        x=xw, y=fr_mean,
        mode="lines",
        line=dict(color=dark_red, width=3),
        name=f"Mean FR (r)<{r_thr})"
    ))
    fig2.add_trace(go.Scatter(
        x=xw, y=ca_mean,
        mode="lines",
        line=dict(color=dark_blue, width=3),
        name=f"Mean Calcium (r)<{r_thr})"
    ))

    fig2.update_layout(
        title=f"Windows with r < {r_thr} (n={good_idx.size})",
        xaxis_title="Time within window (s)",
        yaxis_title="Signal value",
        template="plotly_white",
    )

    # -------------------------
    # 3) subplots, PAGED (list of figures) + secondary y-axis per row
    # -------------------------
    win_t = np.asarray(epoch_corr["win_t"], dtype=float)
    fig_subplots = []
    total = int(good_idx.size)

    per_page = int(per_page)
    if per_page < 1:
        per_page = 1

    for page_start in range(0, total, per_page):
        page_idx = good_idx[page_start:page_start + per_page]
        n = int(page_idx.size)

        # Plotly vertical spacing constraint
        if n <= 1:
            vertical_spacing = 0.0
        else:
            max_vs = 1.0 / (n - 1)
            vertical_spacing = min(float(desired_vertical_spacing), max_vs * 0.98)

        titles = [
            f"WinIdx {idx} | r={corr[idx]:.3f} | {win_t[idx,0]:.2f}s–{win_t[idx,1]:.2f}s"
            for idx in page_idx
        ]

        # ✅ Each row gets a secondary y-axis
        fig_subplot = make_subplots(
            rows=n,
            cols=1,
            shared_xaxes=False,
            vertical_spacing=vertical_spacing,
            subplot_titles=titles,
            specs=[[{"secondary_y": True}] for _ in range(n)]
        )

        for row_i, idx in enumerate(page_idx, start=1):
            fr = np.asarray(window_fr[idx], float).ravel()
            cal = np.asarray(window_cal[idx], float).ravel()

            ws_n_local = fr.size
            xw_local = np.arange(ws_n_local) * (ws_s / ws_n_local)

            # FR on left axis
            fig_subplot.add_trace(
                go.Scatter(
                    x=xw_local, y=fr,
                    mode="lines",
                    line=dict(color=dark_red, width=2),
                    name="FR",
                    showlegend=(row_i == 1)
                ),
                row=row_i, col=1, secondary_y=False
            )

            # Calcium on right axis
            fig_subplot.add_trace(
                go.Scatter(
                    x=xw_local, y=cal,
                    mode="lines",
                    line=dict(color=dark_blue, width=2),
                    name="Calcium",
                    showlegend=(row_i == 1)
                ),
                row=row_i, col=1, secondary_y=True
            )

            fig_subplot.update_xaxes(title_text="Time within window (s)", row=row_i, col=1)
            fig_subplot.update_yaxes(title_text="FR (Hz)", row=row_i, col=1, secondary_y=False)
            fig_subplot.update_yaxes(title_text="ΔF/F", row=row_i, col=1, secondary_y=True)

        page_num = page_start // per_page + 1
        fig_subplot.update_layout(
            width=int(width),
            height=int(row_height) * n,
            template="plotly_white",
            title=f"High-corr windows (r<{r_thr}) — page {page_num} ({page_start+1}-{page_start+n} of {total})",
            margin=dict(t=80, b=40, l=70, r=30),
        )

        fig_subplots.append(fig_subplot)

    return fig1, fig2,fig3, fig_subplots



In [11]:


def plot_corr_vs_meanDiff_by_cell(AllWinMeanDiff, AllWinCor, AllPath=None,
                                  title="Window correlation vs mean FR diff (colored by cell)"):
    fig = go.Figure()

    n_cells = min(len(AllWinMeanDiff), len(AllWinCor))
    for ci in range(n_cells):
        x = np.asarray(AllWinMeanDiff[ci], dtype=float).ravel()
        y = np.asarray(AllWinCor[ci], dtype=float).ravel()

        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() == 0:
            continue

        cell_name = None
        if AllPath is not None and ci < len(AllPath):
            cell_name = str(AllPath[ci])
        else:
            cell_name = f"cell_{ci}"

        fig.add_trace(go.Scatter(
            x=x[m],
            y=y[m],
            mode="markers",
            name=cell_name,      # legend entry (also drives “one color per cell”)
            marker=dict(size=6, opacity=0.75),
            hovertemplate=(
                "cell=%{text}<br>"
                "mean |ΔFR|=%{x:.4g}<br>"
                "r=%{y:.4g}<extra></extra>"
            ),
            text=[cell_name] * int(m.sum()),
        ))

    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis_title="Mean |ΔFR| within window",
        yaxis_title="Pearson r (FR vs calcium)",
        legend_title_text="Cell / trace",
        showlegend=False
            )
    return fig
def plot_meanDiff_as_fn_of_corr_by_cell(AllWinMeanDiff, AllWinCor, AllPath=None,
                                      metric_label="Mean |ΔFR| within window",
                                      metric_hover="mean |ΔFR|",
                                      title="Mean FR diff as a function of window correlation (colored by cell)",
                                      connect_points=False,
                                      sort_by_corr=True):
    """Plot a per-cell scatter of (window correlation -> window metric).

    Each cell contributes multiple points (one per window).
    """
    fig = go.Figure()

    n_cells = min(len(AllWinMeanDiff), len(AllWinCor))
    for ci in range(n_cells):
        x = np.asarray(AllWinCor[ci], dtype=float).ravel()
        y = np.asarray(AllWinMeanDiff[ci], dtype=float).ravel()

        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() == 0:
            continue

        x1 = x[m]
        y1 = y[m]
        if sort_by_corr and x1.size > 1:
            order = np.argsort(x1)
            x1 = x1[order]
            y1 = y1[order]

        if AllPath is not None and ci < len(AllPath):
            cell_name = str(AllPath[ci])
        else:
            cell_name = f"cell_{ci}"

        mode = "lines+markers" if connect_points else "markers"
        fig.add_trace(go.Scatter(
            x=x1,
            y=y1,
            mode=mode,
            name=cell_name,
            marker=dict(size=6, opacity=0.75),
            line=dict(width=1) if connect_points else None,
            hovertemplate=(
                "cell=%{text}<br>"
                "r=%{x:.4g}<br>"
                + metric_hover + "=%{y:.4g}<extra></extra>"
            ),
            text=[cell_name] * int(x1.size),
        ))

    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis_title="Pearson r (FR vs calcium)",
        yaxis_title=metric_label,
        legend_title_text="Cell / trace",
        showlegend=False
    )
    return fig


In [12]:

#SST remove Z
def _load_final_spikes(folder, name=None):
    candidates = []
    if name:
        candidates.append(os.path.join(folder, f"final_spikes_{name}.pkl"))
    candidates.append(os.path.join(folder, "final_spikes.pkl"))
    for p in candidates:
        if os.path.exists(p):
            with open(p, "rb") as f:
                d = pickle.load(f)
            sp = None
            if isinstance(d, dict):
                sp = d.get("spike_indices", None)
                if sp is None:
                    sp = d.get("vm_all_spikes", None)
                if sp is None:
                    sp = d.get("spikes", None)
            if sp is None:
                continue
            sp = np.asarray(sp, dtype=int).ravel().tolist()
            return sp, d, p
    return None, None, None

#SST
AllMeanFR = []
AllBcorr = []
AllCor = []
AllCorMeanWin = []
AllMeanDiffMax = []
AllMeanDiffSeqAbs = []
AllMeanDiffSeq = []
MotorCalTran = []
MotorFrTran = []
RestCalTran = []
RestFrTran = []
AwakeCalTran = []
AwakeFrTran = []
AnstCalTran = []
AnstFrTran = []
MotorPears = []
MotorP_val = []
RestPears = []
RestP_val = []
AwakePears = []
AwakeP_val = []
AnstPears = []
AnstP_val = []
MotorCalDiff = []
MotorFRDiff = []
RestCalDiff = []
RestFRDiff = []
AwakeCalDiff = []
AwakeFRDiff = []
AnstCalDiff = []
AnstFRDiff = []
MotorOpt_r = []
MotorOpt_lag = []
RestOpt_r = []
RestOpt_lag = []
AwakeOpt_r = []
AwakeOpt_lag = []
AnstOpt_r = []
AnstOpt_lag = []
MotorWindow_r = []
MotorWindow_p = []
RestWindow_r = []
RestWindow_p = []
AwakeWindow_r = []
AwakeWindow_p = []
AnstWindow_r = []
AnstWindow_p = []
MotorFRMean = []
RestFRMean=[]
AwakeFRMean = []
AnstFRMean = []
MotorFRdiffMean = []
RestFRdiffMean=[]
AwakeFRdiffMean = []
AnstFRdiffMean = []
AllPath = []
AllWinMeanDiffAbs = []
AllWinMeanDiff = []
AllWinMaxMinDiff = []
AllWinCor =[]


calSR = 30
volSR =500
ws = 10
w_step = 2
fr_sig = 0.01
cal_sig = 0.01
corr_summary_rows = []
for i,l in enumerate(pathPyr):
    calSR= AllCalSR[i]
    #print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    
    
    VolTrace = pd.read_csv(TracePathVol)
    VolTrace = np.array(VolTrace)
    VolTrace = VolTrace.flatten()
    Trace = VolTrace
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(Trace,0)]
    CalTrace = pd.read_csv(TracePathCal)
    CalTrace = np.array(CalTrace)
    CalTrace = CalTrace.flatten()
    VolMask = pd.read_csv(TracePathVolM)
    VolMask = np.array(VolMask)
    VolMask = VolMask.flatten()
    #Trace = VolMask 
    CalMask = pd.read_csv(TracePathCalM)
    CalMask = np.array(CalMask)
    CalMask = CalMask.flatten()
    TraceV = Trace.copy().astype(float)
    TraceC = CalTrace.copy().astype(float)
    motor = motor.to_numpy().astype(float)
    TraceV[~VolMask] = np.nan
    TraceC[~CalMask] = np.nan
    motor[~VolMask] = np.nan
    VolAX = np.linspace(0, (len(TraceV)/500), len(TraceV)) 
    CalAX = np.linspace(0, (len(TraceC)/calSR), len(TraceC))
    if bsPyr[i].lower()=='motor':
        N= ''
        AllPath.append(l)
        spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
        if spikeId is None:
            fpath = os.path.join(l,r'SpikeIdxFinal.csv')
            if os.path.exists(fpath):
                pathSpike = fpath
                spikeId = pd.read_csv(pathSpike)
                spikeId = np.array(spikeId)
                spikeId = spikeId.flatten()
                spikeId = spikeId.tolist()
            else:
                pathSpike = os.path.join(l,r'SpikeIdx.csv')
                print('LOL')
                IspikeId = pd.read_csv(pathSpike)
                IspikeId = np.array(IspikeId)
                IspikeId = IspikeId.flatten()
                IspikeId = IspikeId.tolist()
                VolMask = VolMask.astype(bool)

                # 2. Create a mapping from Old Index -> New Index
                # This creates an array where every index contains the count of "True" frames before it
                new_mapping = np.cumsum(VolMask) - 1 

                # 3. Filter and Map
                # We keep the spike IF the mask was True at that spot...
                # ...AND we convert it to the new index using our mapping.
                spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
        corrDict = fr_corr_and_change_points_scipy(cal_trace=TraceC, fs_ca=calSR,spike_idx_v=spikeId, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=fr_sig,ca_smooth_sigma_s=cal_sig,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
        save_path = os.path.join(l, f"corrDict{N}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path, "wb") as f:
            pickle.dump(corrDict, f)
        epoch_corr = windowed_fr_calcium_correlation(corrDict['fr_on_ca'],corrDict['ca_used'],corrDict['t_ca'],ws_s = ws,step_s=w_step,sigma_fr_s=0.1,sigma_ca_s=0.1)
        
        
        save_path_win = os.path.join(l, f"window_corr{N}_windowlen_{epoch_corr['window_size_s']}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path_win, "wb") as f:
            pickle.dump(epoch_corr, f)
        
        lag_res = lag_scan_pearson(
                x=corrDict['fr_on_ca'],
                y=corrDict['ca_used'],
                fs=calSR,
                max_lag_s=3.0,
                lag_step_s=1/30,
                min_overlap_s=1.0,
                mode="max_pos"
            )
        r_obs = corrDict["pearson_r"]
        r_opt = lag_res["best_r"]
        AllBcorr.append(r_opt)
        AllCor.append(r_obs)
        # --- usage ---
        # epoch_corr = windowed_fr_calcium_correlation(...)
        fig_corr, fig_overlay,fig_scatter,fig_sub,Mean_diff,ccWin = plot_window_corr_results_plotly_top(epoch_corr, r_thr=0.7)
        fig_corr_path_svg = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_corr_path_html = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_diff_path_svg = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_diff_path_html = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_overlay_path_svg = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub, 1):
            fig_subplot_path_svg = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg,fig_subplot_path_html)
        save_plotly_svg_html(fig_corr,fig_corr_path_svg,fig_corr_path_html)
        save_plotly_svg_html(fig_overlay,fig_overlay_path_svg,fig_overlay_path_html)
        save_plotly_svg_html(fig_scatter,fig_diff_path_svg,fig_diff_path_html)
        
        AllWinCor.append(epoch_corr["corr_list"])
        AllMeanFR.append(np.mean(corrDict['fr_on_ca']))
        WinMeanDiffAbs,WinMeanDiff,WinMaxMinDiff = calculate_fr_diff(epoch_corr)
        AllWinMeanDiff.append(WinMeanDiff)
        AllWinMaxMinDiff.append(WinMaxMinDiff)
        AllWinMeanDiffAbs.append(WinMeanDiffAbs)
        AllCorMeanWin.append(np.mean(ccWin))
        AllMeanDiffMax.append(np.mean(WinMaxMinDiff))
        AllMeanDiffSeq.append(np.mean(WinMeanDiff))
        AllMeanDiffSeqAbs.append(np.mean(WinMeanDiffAbs))
        fig_corr_l, fig_overlay_l,fig_scatter_l,fig_sub_l = plot_window_corr_results_plotly_bot(epoch_corr, r_thr=0.1)
        
        fig_overlay_path_svg_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub_l, 1):
            fig_subplot_path_svg_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg_l,fig_subplot_path_html_l)
        
        save_plotly_svg_html(fig_overlay_l,fig_overlay_path_svg_l,fig_overlay_path_html_l)

        fig3 = plot_fr_basic(TraceV,VolAX,spikeId,corrDict['ca_used'],CalAX,corrDict['fr_on_ca'],fr_change_idx=corrDict['fr_change_idx'],pearson_p=corrDict['pearson_p'],pearson_r=corrDict['pearson_r'],path = l)
        fig = add_highcorr_window_markers(fig3, epoch_corr, thr=0.5)

        # 3️⃣ Save the marked version separately
        out_base_marked = os.path.join(l, f"base_FR_full_trace_highcorr_ws_{epoch_corr['window_size_s']}_STS_{w_step}_frSm{fr_sig}_calSm{cal_sig}")
        _save_plotly_fig(fig, out_base_marked, N)
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()

        #calMot,calRes = Split_cal(changeP,TraceV,TraceC,VolAX,CalAX,motor)
        MotorSinglePikeIDX = []
        RestSinglePikeIDX = []
        MotorSinglePikeCalIDX = []
        RestSinglePikeCalIDX = []
        MotorBurstSpikeIDX = []
        RestBurstSpikeIDX = []
        MotorComplexSpikeIDX = []
        RestComplexSpikeIDX = []
        RestFRpath =os.path.join(l,r'FiringRateRest.csv')
        MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        
        dfMot = pd.read_csv(MotorFRpath)
        MotorFR = dfMot['fr'].tolist()
        dfRest = pd.read_csv(RestFRpath)
        RestFR = dfRest['fr'].tolist()
        for k in range(len(MotorFR)):
            
            N = f'm{k}'
            TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
            TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
            VolTrace = pd.read_csv(TracePathVol)
            VolTrace = np.array(VolTrace)
            VolTrace = VolTrace.flatten()
            VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
            CalTrace = pd.read_csv(TracePathCal)
            CalTrace = np.array(CalTrace)
            CalTrace = CalTrace.flatten()
            CalAX = np.linspace(0, (len(CalTrace)/calSR), len(CalTrace))
            spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
            if spikeId is None:
                CorrSPPath =  os.path.join(l,f'spikeTraceMot{k}.csv')
                spikeId = pd.read_csv(CorrSPPath)
                spikeId = np.array(spikeId)
                spikeId = spikeId.flatten()
                spikeId = spikeId.tolist()
            txtPss = os.path.join(l,'SinglS_par.txt')
            corrDict = fr_corr_and_change_points_scipy(cal_trace=CalTrace, fs_ca=calSR,spike_idx_v=spikeId, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=0.0,ca_smooth_sigma_s=0.0,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
            save_path = os.path.join(l, f"corrDict{N}.pkl")

            with open(save_path, "wb") as f:
                pickle.dump(corrDict, f)
            lag_res = lag_scan_pearson(
                x=corrDict['fr_on_ca'],
                y=corrDict['ca_used'],
                fs=calSR,
                max_lag_s=3.0,
                lag_step_s=1/30,
                min_overlap_s=1.0,
                mode="max_pos"
            )
            r_obs = corrDict["pearson_r"]
            r_opt = lag_res["best_r"]
            state_label = (
                'motor' if isinstance(N, str) and N.startswith('m') else
                'rest' if isinstance(N, str) and N.startswith('r') else
                bsPyr[i].lower()
            )
            corr_summary_rows.append({
                'link': l,
                'brain state': str(state_label),
                'optimal lag time': float(lag_res['best_lag_s']),
                'basic r score': float(r_obs),
                'optimal r score': float(r_opt),
            })
            MotorOpt_r.append(lag_res['best_r'])
            MotorOpt_lag.append(lag_res['best_lag_s'])
            win_corr_dict = fr_corr_and_windowed(
                cal_trace=CalTrace, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )
            #AllPath.append(l)
            # Append to state-level lists
            MotorWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
            MotorWindow_p.append(win_corr_dict['window_p_list'])
        for j in range(len(RestFR)):
            
            N = f'r{j}'
            TracePathVol = os.path.join(l,f'volTraceRest{j}.csv')
            TracePathCal = os.path.join(l,f'calTraceRest{j}.csv')
            VolTrace = pd.read_csv(TracePathVol)
            VolTrace = np.array(VolTrace)
            VolTrace = VolTrace.flatten()
            VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
            CalTrace = pd.read_csv(TracePathCal)
            CalTrace = np.array(CalTrace)
            CalTrace = CalTrace.flatten()
            CalAX = np.linspace(0, (len(CalTrace)/calSR), len(CalTrace))
            spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
            if spikeId is None:
                CorrSPPath =  os.path.join(l,f'spikeTraceMot{k}.csv')
                spikeId = pd.read_csv(CorrSPPath)
                spikeId = np.array(spikeId)
                spikeId = spikeId.flatten()
                spikeId = spikeId.tolist()
            txtPss = os.path.join(l,'SinglS_par.txt')
            corrDict = fr_corr_and_change_points_scipy(cal_trace=CalTrace, fs_ca=calSR,spike_idx_v=spikeId, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=0.0,ca_smooth_sigma_s=0.0,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
            save_path = os.path.join(l, f"corrDict{N}.pkl")

            with open(save_path, "wb") as f:
                pickle.dump(corrDict, f)
            lag_res = lag_scan_pearson(
                x=corrDict['fr_on_ca'],
                y=corrDict['ca_used'],
                fs=calSR,
                max_lag_s=3.0,
                lag_step_s=1/30,
                min_overlap_s=1.0,
                mode="max_pos"
            )
            r_obs = corrDict["pearson_r"]
            r_opt = lag_res["best_r"]
            state_label = (
                'motor' if isinstance(N, str) and N.startswith('m') else
                'rest' if isinstance(N, str) and N.startswith('r') else
                bsPyr[i].lower()
            )
            corr_summary_rows.append({
                'link': l,
                'brain state': str(state_label),
                'optimal lag time': float(lag_res['best_lag_s']),
                'basic r score': float(r_obs),
                'optimal r score': float(r_opt),
            })
            RestOpt_r.append(lag_res['best_r'])
            RestOpt_lag.append(lag_res['best_lag_s'])
            win_corr_dict = fr_corr_and_windowed(
                cal_trace=CalTrace, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )
            #AllPath.append(l)
            # Append to state-level lists
            RestWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
            RestWindow_p.append(win_corr_dict['window_p_list'])
    elif bsPyr[i].lower()=='awake':
        N = ''
        AllPath.append(l)
        spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
        if spikeId is None:
            fpath = os.path.join(l,r'SpikeIdxFinal.csv')
            if os.path.exists(fpath):
                pathSpike = fpath
                spikeId = pd.read_csv(pathSpike)
                spikeId = np.array(spikeId)
                spikeId = spikeId.flatten()
                spikeId = spikeId.tolist()
            else:
                pathSpike = os.path.join(l,r'SpikeIdx.csv')
                print('LOL')
                IspikeId = pd.read_csv(pathSpike)
                IspikeId = np.array(IspikeId)
                IspikeId = IspikeId.flatten()
                IspikeId = IspikeId.tolist()
                VolMask = VolMask.astype(bool)

                # 2. Create a mapping from Old Index -> New Index
                # This creates an array where every index contains the count of "True" frames before it
                new_mapping = np.cumsum(VolMask) - 1 

                # 3. Filter and Map
                # We keep the spike IF the mask was True at that spot...
                # ...AND we convert it to the new index using our mapping.
                spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
        txtPss = os.path.join(l,f'SinglS_par_frSm{fr_sig}_calSm{cal_sig}.txt')
        corrDict = fr_corr_and_change_points_scipy(cal_trace=TraceC, fs_ca=calSR,spike_idx_v=spikeId, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=fr_sig,ca_smooth_sigma_s=cal_sig,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
        epoch_corr = windowed_fr_calcium_correlation(corrDict['fr_on_ca'],corrDict['ca_used'],corrDict['t_ca'],ws_s = ws,step_s=w_step,sigma_fr_s=0.1,sigma_ca_s=0.1)
        
        
        save_path_win = os.path.join(l, f"window_corr{N}_windowlen_{epoch_corr['window_size_s']}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path_win, "wb") as f:
            pickle.dump(epoch_corr, f)
        save_path = os.path.join(l, f"corrDict{N}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path, "wb") as f:
                pickle.dump(corrDict, f)
        # epoch_corr = windowed_fr_calcium_correlation(...)
        fig_corr, fig_overlay,fig_scatter,fig_sub,Mean_diff,ccWin = plot_window_corr_results_plotly_top(epoch_corr, r_thr=0.7)
        fig_corr_path_svg = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_corr_path_html = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_diff_path_svg = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_diff_path_html = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_overlay_path_svg = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub, 1):
            fig_subplot_path_svg = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg,fig_subplot_path_html)
        save_plotly_svg_html(fig_corr,fig_corr_path_svg,fig_corr_path_html)
        save_plotly_svg_html(fig_overlay,fig_overlay_path_svg,fig_overlay_path_html)
        save_plotly_svg_html(fig_scatter,fig_diff_path_svg,fig_diff_path_html)
        
        AllWinCor.append(epoch_corr["corr_list"])
        WinMeanDiffAbs,WinMeanDiff,WinMaxMinDiff = calculate_fr_diff(epoch_corr)
        AllWinMeanDiff.append(WinMeanDiff)
        AllWinMaxMinDiff.append(WinMaxMinDiff)
        AllWinMeanDiffAbs.append(WinMeanDiffAbs)
        AllMeanDiffMax.append(np.mean(WinMaxMinDiff))
        AllMeanDiffSeq.append(np.mean(WinMeanDiff))
        AllMeanDiffSeqAbs.append(np.mean(WinMeanDiffAbs))
        AllCorMeanWin.append(np.mean(ccWin))
        AllMeanFR.append(np.mean(corrDict['fr_on_ca']))


        fig_corr_l, fig_overlay_l,fig_scatter_l,fig_sub_l = plot_window_corr_results_plotly_bot(epoch_corr, r_thr=0.1)
        
        fig_overlay_path_svg_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub_l, 1):
            fig_subplot_path_svg_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg_l,fig_subplot_path_html_l)
        
        save_plotly_svg_html(fig_overlay_l,fig_overlay_path_svg_l,fig_overlay_path_html_l)
        
        lag_res = lag_scan_pearson(
            x=corrDict['fr_on_ca'],
            y=corrDict['ca_used'],
            fs=calSR,
            max_lag_s=3.0,
            lag_step_s=1/30,
            min_overlap_s=1.0,
            mode="max_pos"
        )
        r_obs = corrDict["pearson_r"]
        r_opt = lag_res["best_r"]
        AllBcorr.append(r_opt)
        AllCor.append(r_obs)
        state_label = (
            'motor' if isinstance(N, str) and N.startswith('m') else
            'rest' if isinstance(N, str) and N.startswith('r') else
            bsPyr[i].lower()
        )
        corr_summary_rows.append({
            'link': l,
            'brain state': str(state_label),
            'optimal lag time': float(lag_res['best_lag_s']),
            'basic r score': float(r_obs),
            'optimal r score': float(r_opt),
        })
        calEvnDic = optionB_ca_event_windows_compare_fr(ca_dff=TraceC,fs_ca=calSR,voltage_trace=TraceV,fs_v=500,ca_smooth_sigma_s=0.07,spike_idx_v=spikeId,save_path=l)
        # 1) Event-triggered plot (uses ca_event_traces if you saved them; otherwise falls back to onsets)
        fig1 = plot_event_triggered_fr_and_ca(calEvnDic,l)

        # 2) Whole-trace plot (uses ONLY ca_onsets_used_idx from the dict; boxes end at threshold drop)
        fig2 = plot_whole_trace_voltage_calcium_with_events(voltage_trace=TraceV, VolXaX=VolAX,ca_dff=TraceC, CalXax=CalAX,spike_idx_v=spikeId,calciumeventsDic=calEvnDic,out_dir=l)
        #cur_SS_Vol,cur_SS_Cal,cal_SS_idx = get_Vol_Cal_SS(single_spikes_list,VolTrace,CalTrace,VolAX,CalAX,save_path=txtPss)
        fig3 = plot_fr_basic(TraceV,VolAX,spikeId,corrDict['ca_used'],CalAX,corrDict['fr_on_ca'],fr_change_idx=corrDict['fr_change_idx'],pearson_p=corrDict['pearson_p'],pearson_r=corrDict['pearson_r'],path = l)
        fig = add_highcorr_window_markers(fig3, epoch_corr, thr=0.5)

        # 3️⃣ Save the marked version separately
        out_base_marked = os.path.join(l, f"base_FR_full_trace_highcorr_ws_{epoch_corr['window_size_s']}_frSm{fr_sig}_calSm{cal_sig}")
        _save_plotly_fig(fig, out_base_marked, N)
        TracePathCal = os.path.join(l,'changepoint.csv')
        # observed zero-lag correlation
        r_obs = corrDict["pearson_r"]

        # optimal lag correlation (from lag_scan_pearson)
        AwakeOpt_r.append(lag_res['best_r'])
        AwakeOpt_lag.append(lag_res['best_lag_s'])
        win_corr_dict = fr_corr_and_windowed(
                cal_trace=TraceC, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )

            # Append to state-level lists
        AwakeWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
        AwakeWindow_p.append(win_corr_dict['window_p_list'])
    elif bsPyr[i].lower()=='anst':
        N = ''
        AllPath.append(l)
        spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
        if spikeId is None:
            fpath = os.path.join(l,r'SpikeIdxFinal.csv')
            if os.path.exists(fpath):
                pathSpike = fpath
                spikeId = pd.read_csv(pathSpike)
                spikeId = np.array(spikeId)
                spikeId = spikeId.flatten()
                spikeId = spikeId.tolist()
            else:
                pathSpike = os.path.join(l,r'SpikeIdx.csv')
                print('LOL')
                IspikeId = pd.read_csv(pathSpike)
                IspikeId = np.array(IspikeId)
                IspikeId = IspikeId.flatten()
                IspikeId = IspikeId.tolist()
                VolMask = VolMask.astype(bool)

                # 2. Create a mapping from Old Index -> New Index
                # This creates an array where every index contains the count of "True" frames before it
                new_mapping = np.cumsum(VolMask) - 1 

                # 3. Filter and Map
                # We keep the spike IF the mask was True at that spot...
                # ...AND we convert it to the new index using our mapping.
                spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
        corrDict = fr_corr_and_change_points_scipy(
            cal_trace=TraceC, fs_ca=calSR, spike_idx_v=spikeId, fs_v=500,
            fr_bin_s=0.1,
            fr_smooth_sigma_s=fr_sig,
            ca_smooth_sigma_s=cal_sig,
            change_method="mad", change_k=3.0,
            min_separation_s=0.3,
        )
        save_path = os.path.join(l, f"corrDict{N}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path, "wb") as f:
                pickle.dump(corrDict, f)
        epoch_corr = windowed_fr_calcium_correlation(corrDict['fr_on_ca'],corrDict['ca_used'],corrDict['t_ca'],ws_s = ws,step_s=w_step,sigma_fr_s=0.1,sigma_ca_s=0.1)
        
        
        save_path_win = os.path.join(l, f"window_corr{N}_windowlen_{epoch_corr['window_size_s']}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path_win, "wb") as f:
            pickle.dump(epoch_corr, f)

        fig_corr, fig_overlay,fig_scatter,fig_sub,Mean_diff,ccWin = plot_window_corr_results_plotly_top(epoch_corr, r_thr=0.7)
        fig_corr_path_svg = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_corr_path_html = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_diff_path_svg = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_diff_path_html = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_overlay_path_svg = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub, 1):
            fig_subplot_path_svg = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg,fig_subplot_path_html)
        save_plotly_svg_html(fig_corr,fig_corr_path_svg,fig_corr_path_html)
        save_plotly_svg_html(fig_overlay,fig_overlay_path_svg,fig_overlay_path_html)
        save_plotly_svg_html(fig_scatter,fig_diff_path_svg,fig_diff_path_html)
        
        AllWinCor.append(epoch_corr["corr_list"])
        WinMeanDiffAbs,WinMeanDiff,WinMaxMinDiff = calculate_fr_diff(epoch_corr)
        AllWinMeanDiff.append(WinMeanDiff)
        AllWinMaxMinDiff.append(WinMaxMinDiff)
        AllWinMeanDiffAbs.append(WinMeanDiffAbs)
        AllMeanDiffMax.append(np.mean(WinMaxMinDiff))
        AllMeanDiffSeq.append(np.mean(WinMeanDiff))
        AllMeanDiffSeqAbs.append(np.mean(WinMeanDiffAbs))
        AllCorMeanWin.append(np.mean(ccWin))
        AllMeanFR.append(np.mean(corrDict['fr_on_ca']))


        fig_corr_l, fig_overlay_l,fig_scatter_l,fig_sub_l = plot_window_corr_results_plotly_bot(epoch_corr, r_thr=0.1)
        
        fig_overlay_path_svg_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub_l, 1):
            fig_subplot_path_svg_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg_l,fig_subplot_path_html_l)
        
        save_plotly_svg_html(fig_overlay_l,fig_overlay_path_svg_l,fig_overlay_path_html_l)



        lag_res = lag_scan_pearson(
            x=corrDict['fr_on_ca'],
            y=corrDict['ca_used'],
            fs=calSR,
            max_lag_s=3.0,
            lag_step_s=1/30,
            min_overlap_s=1.0,
            mode="max_pos"
        )
        r_obs = corrDict["pearson_r"]
        r_opt = lag_res["best_r"]
        AllBcorr.append(r_opt)
        AllCor.append(r_obs)
        state_label = (
            'motor' if isinstance(N, str) and N.startswith('m') else
            'rest' if isinstance(N, str) and N.startswith('r') else
            bsPyr[i].lower()
        )
        corr_summary_rows.append({
            'link': l,
            'brain state': str(state_label),
            'optimal lag time': float(lag_res['best_lag_s']),
            'basic r score': float(r_obs),
            'optimal r score': float(r_opt),
        })
        calEvnDic = optionB_ca_event_windows_compare_fr(ca_dff=TraceC,fs_ca=calSR,voltage_trace=TraceV,fs_v=500,ca_smooth_sigma_s=0.07,spike_idx_v=spikeId,save_path=l)
        # 1) Event-triggered plot (uses ca_event_traces if you saved them; otherwise falls back to onsets)
        fig1 = plot_event_triggered_fr_and_ca(calEvnDic,l)

        # 2) Whole-trace plot (uses ONLY ca_onsets_used_idx from the dict; boxes end at threshold drop)
        fig2 = plot_whole_trace_voltage_calcium_with_events(voltage_trace=TraceV, VolXaX=VolAX,ca_dff=TraceC, CalXax=CalAX,spike_idx_v=spikeId,calciumeventsDic=calEvnDic,out_dir=l)
        #cur_SS_Vol,cur_SS_Cal,cal_SS_idx = get_Vol_Cal_SS(single_spikes_list,VolTrace,CalTrace,VolAX,CalAX,save_path=txtPss)
        fig3 = plot_fr_basic(TraceV,VolAX,spikeId,corrDict['ca_used'],CalAX,corrDict['fr_on_ca'],fr_change_idx=corrDict['fr_change_idx'],pearson_p=corrDict['pearson_p'],pearson_r=corrDict['pearson_r'],path = l)
        fig = add_highcorr_window_markers(fig3, epoch_corr, thr=0.5)

        # 3️⃣ Save the marked version separately
        out_base_marked = os.path.join(l, f"base_FR_full_trace_highcorr_ws_{epoch_corr['window_size_s']}_frSm{fr_sig}_calSm{cal_sig}")
        _save_plotly_fig(fig, out_base_marked, N)
        TracePathCal = os.path.join(l,'changepoint.csv')
        # observed zero-lag correlation
        r_obs = corrDict["pearson_r"]

        # optimal lag correlation (from lag_scan_pearson)
        AnstOpt_r.append(lag_res['best_r'])
        AnstOpt_lag.append(lag_res['best_lag_s'])
        win_corr_dict = fr_corr_and_windowed(
                cal_trace=TraceC, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )

            # Append to state-level lists
        AnstWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
        AnstWindow_p.append(win_corr_dict['window_p_list'])
out_csv = f"Z:\\Adam-Lab-Shared\\Data\\Michal_Rubin\\data summery\\2026\\SST_corralation_sUMMARY_frSm{fr_sig}_calSm{cal_sig}.csv"
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df_summary = pd.DataFrame(corr_summary_rows)
cols = ['link','brain state','optimal lag time','basic r score','optimal r score']
df_summary = df_summary[[c for c in cols if c in df_summary.columns]]
df_summary.to_csv(out_csv, index=False)
# print(f"Saved correlation summary: {out_csv} ({len(corr_summary_rows)} rows)")

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\kaleido\scopes\base.py:185: DeprecationWarning:

setDaemon() is deprecated, set the daemon attribute instead

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\l

⚠️ No usable events for FR plot (fr_trials is None/empty).
   n_events_total: 0
   n_events_used : 0


c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\owner\AppData\Local\Temp\ipykernel_19944\438599033.py:278: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning:

invalid value encountered in divide



⚠️ No usable events for FR plot (fr_trials is None/empty).
   n_events_total: 0
   n_events_used : 0


c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\owner\AppData\Local\Temp\ipykernel_19944\438599033.py:278: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning:

invalid value encountered in divide



...

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\num

⚠️ No usable events for FR plot (fr_trials is None/empty).
   n_events_total: 0
   n_events_used : 0


c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\num

⚠️ No usable events for FR plot (fr_trials is None/empty).
   n_events_total: 0
   n_events_used : 0


c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide



In [51]:


def _load_final_spikes(folder, name=None):
    candidates = []
    if name:
        candidates.append(os.path.join(folder, f"final_spikes_{name}.pkl"))
    candidates.append(os.path.join(folder, "final_spikes.pkl"))
    for p in candidates:
        if os.path.exists(p):
            with open(p, "rb") as f:
                d = pickle.load(f)
            sp = None
            if isinstance(d, dict):
                sp = d.get("spike_indices", None)
                if sp is None:
                    sp = d.get("vm_all_spikes", None)
                if sp is None:
                    sp = d.get("spikes", None)
            if sp is None:
                continue
            sp = np.asarray(sp, dtype=int).ravel().tolist()
            return sp, d, p
    return None, None, None

#SST
AllMeanFR = []
AllBcorr = []
AllCor = []
AllCorMeanWin = []
AllMeanDiffMax = []
AllMeanDiffSeqAbs = []
AllMeanDiffSeq = []
MotorCalTran = []
MotorFrTran = []
RestCalTran = []
RestFrTran = []
AwakeCalTran = []
AwakeFrTran = []
AnstCalTran = []
AnstFrTran = []
MotorPears = []
MotorP_val = []
RestPears = []
RestP_val = []
AwakePears = []
AwakeP_val = []
AnstPears = []
AnstP_val = []
MotorCalDiff = []
MotorFRDiff = []
RestCalDiff = []
RestFRDiff = []
AwakeCalDiff = []
AwakeFRDiff = []
AnstCalDiff = []
AnstFRDiff = []
MotorOpt_r = []
MotorOpt_lag = []
RestOpt_r = []
RestOpt_lag = []
AwakeOpt_r = []
AwakeOpt_lag = []
AnstOpt_r = []
AnstOpt_lag = []
MotorWindow_r = []
MotorWindow_p = []
RestWindow_r = []
RestWindow_p = []
AwakeWindow_r = []
AwakeWindow_p = []
AnstWindow_r = []
AnstWindow_p = []
MotorFRMean = []
RestFRMean=[]
AwakeFRMean = []
AnstFRMean = []
MotorFRdiffMean = []
RestFRdiffMean=[]
AwakeFRdiffMean = []
AnstFRdiffMean = []
AllPath = []
AllWinMeanDiffAbs = []
AllWinMeanDiff = []
AllWinMaxMinDiff = []
AllWinCor =[]


calSR = 30
volSR =500
ws = 10
w_step = 2
fr_sig = 0.15
cal_sig = 0.3
corr_summary_rows = []
for i,l in enumerate(pathPyr):
    calSR= AllCalSR[i]
    #print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    
    
    VolTrace = pd.read_csv(TracePathVol)
    VolTrace = np.array(VolTrace)
    VolTrace = VolTrace.flatten()
    Trace = VolTrace
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(Trace,0)]
    CalTrace = pd.read_csv(TracePathCal)
    CalTrace = np.array(CalTrace)
    CalTrace = CalTrace.flatten()
    VolMask = pd.read_csv(TracePathVolM)
    VolMask = np.array(VolMask)
    VolMask = VolMask.flatten()
    #Trace = VolMask 
    CalMask = pd.read_csv(TracePathCalM)
    CalMask = np.array(CalMask)
    CalMask = CalMask.flatten()
    TraceV = Trace[VolMask]
    TraceC = CalTrace[CalMask]
    motor= motor[VolMask]
    VolAX = np.linspace(0, (len(TraceV)/500), len(TraceV)) 
    CalAX = np.linspace(0, (len(TraceC)/calSR), len(TraceC))
    if bsPyr[i].lower()=='motor':
        N= ''
        AllPath.append(l)
        spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
        if spikeId is None:
            fpath = os.path.join(l,r'SpikeIdxFinal.csv')
            if os.path.exists(fpath):
                pathSpike = fpath
                spikeId = pd.read_csv(pathSpike)
                spikeId = np.array(spikeId)
                spikeId = spikeId.flatten()
                spikeId = spikeId.tolist()
            else:
                pathSpike = os.path.join(l,r'SpikeIdx.csv')
                print('LOL')
                IspikeId = pd.read_csv(pathSpike)
                IspikeId = np.array(IspikeId)
                IspikeId = IspikeId.flatten()
                IspikeId = IspikeId.tolist()
                VolMask = VolMask.astype(bool)

                # 2. Create a mapping from Old Index -> New Index
                # This creates an array where every index contains the count of "True" frames before it
                new_mapping = np.cumsum(VolMask) - 1 

                # 3. Filter and Map
                # We keep the spike IF the mask was True at that spot...
                # ...AND we convert it to the new index using our mapping.
                spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
        corrDict = fr_corr_and_change_points_scipy(cal_trace=TraceC, fs_ca=calSR,spike_idx_v=spikeId, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=fr_sig,ca_smooth_sigma_s=cal_sig,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
        save_path = os.path.join(l, f"corrDict{N}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path, "wb") as f:
            pickle.dump(corrDict, f)
        epoch_corr = windowed_fr_calcium_correlation(corrDict['fr_on_ca'],corrDict['ca_used'],corrDict['t_ca'],ws_s = ws,step_s=w_step,sigma_fr_s=0.1,sigma_ca_s=0.1)
        
        
        save_path_win = os.path.join(l, f"window_corr{N}_windowlen_{epoch_corr['window_size_s']}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path_win, "wb") as f:
            pickle.dump(epoch_corr, f)
        
        lag_res = lag_scan_pearson(
                x=corrDict['fr_on_ca'],
                y=corrDict['ca_used'],
                fs=calSR,
                max_lag_s=3.0,
                lag_step_s=1/30,
                min_overlap_s=1.0,
                mode="max_pos"
            )
        r_obs = corrDict["pearson_r"]
        r_opt = lag_res["best_r"]
        AllBcorr.append(r_opt)
        AllCor.append(r_obs)
        # --- usage ---
        # epoch_corr = windowed_fr_calcium_correlation(...)
        fig_corr, fig_overlay,fig_scatter,fig_sub,Mean_diff,ccWin = plot_window_corr_results_plotly_top(epoch_corr, r_thr=0.7)
        fig_corr_path_svg = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_corr_path_html = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_diff_path_svg = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_diff_path_html = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_overlay_path_svg = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub, 1):
            fig_subplot_path_svg = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg,fig_subplot_path_html)
        save_plotly_svg_html(fig_corr,fig_corr_path_svg,fig_corr_path_html)
        save_plotly_svg_html(fig_overlay,fig_overlay_path_svg,fig_overlay_path_html)
        save_plotly_svg_html(fig_scatter,fig_diff_path_svg,fig_diff_path_html)
        
        AllWinCor.append(epoch_corr["corr_list"])
        AllMeanFR.append(np.mean(corrDict['fr_on_ca']))
        WinMeanDiffAbs,WinMeanDiff,WinMaxMinDiff = calculate_fr_diff(epoch_corr)
        AllWinMeanDiff.append(WinMeanDiff)
        AllWinMaxMinDiff.append(WinMaxMinDiff)
        AllWinMeanDiffAbs.append(WinMeanDiffAbs)
        AllCorMeanWin.append(np.mean(ccWin))
        AllMeanDiffMax.append(np.mean(WinMaxMinDiff))
        AllMeanDiffSeq.append(np.mean(WinMeanDiff))
        AllMeanDiffSeqAbs.append(np.mean(WinMeanDiffAbs))
        fig_corr_l, fig_overlay_l,fig_scatter_l,fig_sub_l = plot_window_corr_results_plotly_bot(epoch_corr, r_thr=0.1)
        
        fig_overlay_path_svg_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub_l, 1):
            fig_subplot_path_svg_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg_l,fig_subplot_path_html_l)
        
        save_plotly_svg_html(fig_overlay_l,fig_overlay_path_svg_l,fig_overlay_path_html_l)

        fig3 = plot_fr_basic(TraceV,VolAX,spikeId,corrDict['ca_used'],CalAX,corrDict['fr_on_ca'],fr_change_idx=corrDict['fr_change_idx'],pearson_p=corrDict['pearson_p'],pearson_r=corrDict['pearson_r'],path = l)
        fig = add_highcorr_window_markers(fig3, epoch_corr, thr=0.5)

        # 3️⃣ Save the marked version separately
        out_base_marked = os.path.join(l, f"base_FR_full_trace_highcorr_ws_{epoch_corr['window_size_s']}_STS_{w_step}_frSm{fr_sig}_calSm{cal_sig}")
        _save_plotly_fig(fig, out_base_marked, N)
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()

        #calMot,calRes = Split_cal(changeP,TraceV,TraceC,VolAX,CalAX,motor)
        MotorSinglePikeIDX = []
        RestSinglePikeIDX = []
        MotorSinglePikeCalIDX = []
        RestSinglePikeCalIDX = []
        MotorBurstSpikeIDX = []
        RestBurstSpikeIDX = []
        MotorComplexSpikeIDX = []
        RestComplexSpikeIDX = []
        RestFRpath =os.path.join(l,r'FiringRateRest.csv')
        MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        
        dfMot = pd.read_csv(MotorFRpath)
        MotorFR = dfMot['fr'].tolist()
        dfRest = pd.read_csv(RestFRpath)
        RestFR = dfRest['fr'].tolist()
        for k in range(len(MotorFR)):
            
            N = f'm{k}'
            TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
            TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
            VolTrace = pd.read_csv(TracePathVol)
            VolTrace = np.array(VolTrace)
            VolTrace = VolTrace.flatten()
            VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
            CalTrace = pd.read_csv(TracePathCal)
            CalTrace = np.array(CalTrace)
            CalTrace = CalTrace.flatten()
            CalAX = np.linspace(0, (len(CalTrace)/calSR), len(CalTrace))
            spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
            if spikeId is None:
                CorrSPPath =  os.path.join(l,f'spikeTraceMot{k}.csv')
                spikeId = pd.read_csv(CorrSPPath)
                spikeId = np.array(spikeId)
                spikeId = spikeId.flatten()
                spikeId = spikeId.tolist()
            txtPss = os.path.join(l,'SinglS_par.txt')
            corrDict = fr_corr_and_change_points_scipy(cal_trace=CalTrace, fs_ca=calSR,spike_idx_v=spikeId, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=0.0,ca_smooth_sigma_s=0.0,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
            save_path = os.path.join(l, f"corrDict{N}.pkl")

            with open(save_path, "wb") as f:
                pickle.dump(corrDict, f)
            lag_res = lag_scan_pearson(
                x=corrDict['fr_on_ca'],
                y=corrDict['ca_used'],
                fs=calSR,
                max_lag_s=3.0,
                lag_step_s=1/30,
                min_overlap_s=1.0,
                mode="max_pos"
            )
            r_obs = corrDict["pearson_r"]
            r_opt = lag_res["best_r"]
            state_label = (
                'motor' if isinstance(N, str) and N.startswith('m') else
                'rest' if isinstance(N, str) and N.startswith('r') else
                bsPyr[i].lower()
            )
            corr_summary_rows.append({
                'link': l,
                'brain state': str(state_label),
                'optimal lag time': float(lag_res['best_lag_s']),
                'basic r score': float(r_obs),
                'optimal r score': float(r_opt),
            })
            MotorOpt_r.append(lag_res['best_r'])
            MotorOpt_lag.append(lag_res['best_lag_s'])
            win_corr_dict = fr_corr_and_windowed(
                cal_trace=CalTrace, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )
            #AllPath.append(l)
            # Append to state-level lists
            MotorWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
            MotorWindow_p.append(win_corr_dict['window_p_list'])
        for j in range(len(RestFR)):
            
            N = f'r{j}'
            TracePathVol = os.path.join(l,f'volTraceRest{j}.csv')
            TracePathCal = os.path.join(l,f'calTraceRest{j}.csv')
            VolTrace = pd.read_csv(TracePathVol)
            VolTrace = np.array(VolTrace)
            VolTrace = VolTrace.flatten()
            VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
            CalTrace = pd.read_csv(TracePathCal)
            CalTrace = np.array(CalTrace)
            CalTrace = CalTrace.flatten()
            CalAX = np.linspace(0, (len(CalTrace)/calSR), len(CalTrace))
            spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
            if spikeId is None:
                CorrSPPath =  os.path.join(l,f'spikeTraceMot{k}.csv')
                spikeId = pd.read_csv(CorrSPPath)
                spikeId = np.array(spikeId)
                spikeId = spikeId.flatten()
                spikeId = spikeId.tolist()
            txtPss = os.path.join(l,'SinglS_par.txt')
            corrDict = fr_corr_and_change_points_scipy(cal_trace=CalTrace, fs_ca=calSR,spike_idx_v=spikeId, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=0.0,ca_smooth_sigma_s=0.0,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
            save_path = os.path.join(l, f"corrDict{N}.pkl")

            with open(save_path, "wb") as f:
                pickle.dump(corrDict, f)
            lag_res = lag_scan_pearson(
                x=corrDict['fr_on_ca'],
                y=corrDict['ca_used'],
                fs=calSR,
                max_lag_s=3.0,
                lag_step_s=1/30,
                min_overlap_s=1.0,
                mode="max_pos"
            )
            r_obs = corrDict["pearson_r"]
            r_opt = lag_res["best_r"]
            state_label = (
                'motor' if isinstance(N, str) and N.startswith('m') else
                'rest' if isinstance(N, str) and N.startswith('r') else
                bsPyr[i].lower()
            )
            corr_summary_rows.append({
                'link': l,
                'brain state': str(state_label),
                'optimal lag time': float(lag_res['best_lag_s']),
                'basic r score': float(r_obs),
                'optimal r score': float(r_opt),
            })
            RestOpt_r.append(lag_res['best_r'])
            RestOpt_lag.append(lag_res['best_lag_s'])
            win_corr_dict = fr_corr_and_windowed(
                cal_trace=CalTrace, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )
            #AllPath.append(l)
            # Append to state-level lists
            RestWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
            RestWindow_p.append(win_corr_dict['window_p_list'])
    elif bsPyr[i].lower()=='awake':
        N = ''
        AllPath.append(l)
        spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
        if spikeId is None:
            fpath = os.path.join(l,r'SpikeIdxFinal.csv')
            if os.path.exists(fpath):
                pathSpike = fpath
                spikeId = pd.read_csv(pathSpike)
                spikeId = np.array(spikeId)
                spikeId = spikeId.flatten()
                spikeId = spikeId.tolist()
            else:
                pathSpike = os.path.join(l,r'SpikeIdx.csv')
                print('LOL')
                IspikeId = pd.read_csv(pathSpike)
                IspikeId = np.array(IspikeId)
                IspikeId = IspikeId.flatten()
                IspikeId = IspikeId.tolist()
                VolMask = VolMask.astype(bool)

                # 2. Create a mapping from Old Index -> New Index
                # This creates an array where every index contains the count of "True" frames before it
                new_mapping = np.cumsum(VolMask) - 1 

                # 3. Filter and Map
                # We keep the spike IF the mask was True at that spot...
                # ...AND we convert it to the new index using our mapping.
                spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
        txtPss = os.path.join(l,f'SinglS_par_frSm{fr_sig}_calSm{cal_sig}.txt')
        corrDict = fr_corr_and_change_points_scipy(cal_trace=TraceC, fs_ca=calSR,spike_idx_v=spikeId, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=fr_sig,ca_smooth_sigma_s=cal_sig,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
        epoch_corr = windowed_fr_calcium_correlation(corrDict['fr_on_ca'],corrDict['ca_used'],corrDict['t_ca'],ws_s = ws,step_s=w_step,sigma_fr_s=0.1,sigma_ca_s=0.1)
        
        
        save_path_win = os.path.join(l, f"window_corr{N}_windowlen_{epoch_corr['window_size_s']}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path_win, "wb") as f:
            pickle.dump(epoch_corr, f)
        save_path = os.path.join(l, f"corrDict{N}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path, "wb") as f:
                pickle.dump(corrDict, f)
        # epoch_corr = windowed_fr_calcium_correlation(...)
        fig_corr, fig_overlay,fig_scatter,fig_sub,Mean_diff,ccWin = plot_window_corr_results_plotly_top(epoch_corr, r_thr=0.7)
        fig_corr_path_svg = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_corr_path_html = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_diff_path_svg = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_diff_path_html = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_overlay_path_svg = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub, 1):
            fig_subplot_path_svg = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg,fig_subplot_path_html)
        save_plotly_svg_html(fig_corr,fig_corr_path_svg,fig_corr_path_html)
        save_plotly_svg_html(fig_overlay,fig_overlay_path_svg,fig_overlay_path_html)
        save_plotly_svg_html(fig_scatter,fig_diff_path_svg,fig_diff_path_html)
        
        AllWinCor.append(epoch_corr["corr_list"])
        WinMeanDiffAbs,WinMeanDiff,WinMaxMinDiff = calculate_fr_diff(epoch_corr)
        AllWinMeanDiff.append(WinMeanDiff)
        AllWinMaxMinDiff.append(WinMaxMinDiff)
        AllWinMeanDiffAbs.append(WinMeanDiffAbs)
        AllMeanDiffMax.append(np.mean(WinMaxMinDiff))
        AllMeanDiffSeq.append(np.mean(WinMeanDiff))
        AllMeanDiffSeqAbs.append(np.mean(WinMeanDiffAbs))
        AllCorMeanWin.append(np.mean(ccWin))
        AllMeanFR.append(np.mean(corrDict['fr_on_ca']))


        fig_corr_l, fig_overlay_l,fig_scatter_l,fig_sub_l = plot_window_corr_results_plotly_bot(epoch_corr, r_thr=0.1)
        
        fig_overlay_path_svg_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub_l, 1):
            fig_subplot_path_svg_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg_l,fig_subplot_path_html_l)
        
        save_plotly_svg_html(fig_overlay_l,fig_overlay_path_svg_l,fig_overlay_path_html_l)
        
        lag_res = lag_scan_pearson(
            x=corrDict['fr_on_ca'],
            y=corrDict['ca_used'],
            fs=calSR,
            max_lag_s=3.0,
            lag_step_s=1/30,
            min_overlap_s=1.0,
            mode="max_pos"
        )
        r_obs = corrDict["pearson_r"]
        r_opt = lag_res["best_r"]
        AllBcorr.append(r_opt)
        AllCor.append(r_obs)
        state_label = (
            'motor' if isinstance(N, str) and N.startswith('m') else
            'rest' if isinstance(N, str) and N.startswith('r') else
            bsPyr[i].lower()
        )
        corr_summary_rows.append({
            'link': l,
            'brain state': str(state_label),
            'optimal lag time': float(lag_res['best_lag_s']),
            'basic r score': float(r_obs),
            'optimal r score': float(r_opt),
        })
        calEvnDic = optionB_ca_event_windows_compare_fr(ca_dff=TraceC,fs_ca=calSR,voltage_trace=TraceV,fs_v=500,ca_smooth_sigma_s=0.07,spike_idx_v=spikeId,save_path=l)
        # 1) Event-triggered plot (uses ca_event_traces if you saved them; otherwise falls back to onsets)
        fig1 = plot_event_triggered_fr_and_ca(calEvnDic,l)

        # 2) Whole-trace plot (uses ONLY ca_onsets_used_idx from the dict; boxes end at threshold drop)
        fig2 = plot_whole_trace_voltage_calcium_with_events(voltage_trace=TraceV, VolXaX=VolAX,ca_dff=TraceC, CalXax=CalAX,spike_idx_v=spikeId,calciumeventsDic=calEvnDic,out_dir=l)
        #cur_SS_Vol,cur_SS_Cal,cal_SS_idx = get_Vol_Cal_SS(single_spikes_list,VolTrace,CalTrace,VolAX,CalAX,save_path=txtPss)
        fig3 = plot_fr_basic(TraceV,VolAX,spikeId,corrDict['ca_used'],CalAX,corrDict['fr_on_ca'],fr_change_idx=corrDict['fr_change_idx'],pearson_p=corrDict['pearson_p'],pearson_r=corrDict['pearson_r'],path = l)
        fig = add_highcorr_window_markers(fig3, epoch_corr, thr=0.5)

        # 3️⃣ Save the marked version separately
        out_base_marked = os.path.join(l, f"base_FR_full_trace_highcorr_ws_{epoch_corr['window_size_s']}_frSm{fr_sig}_calSm{cal_sig}")
        _save_plotly_fig(fig, out_base_marked, N)
        TracePathCal = os.path.join(l,'changepoint.csv')
        # observed zero-lag correlation
        r_obs = corrDict["pearson_r"]

        # optimal lag correlation (from lag_scan_pearson)
        AwakeOpt_r.append(lag_res['best_r'])
        AwakeOpt_lag.append(lag_res['best_lag_s'])
        win_corr_dict = fr_corr_and_windowed(
                cal_trace=TraceC, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )

            # Append to state-level lists
        AwakeWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
        AwakeWindow_p.append(win_corr_dict['window_p_list'])
    elif bsPyr[i].lower()=='anst':
        N = ''
        AllPath.append(l)
        spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
        if spikeId is None:
            fpath = os.path.join(l,r'SpikeIdxFinal.csv')
            if os.path.exists(fpath):
                pathSpike = fpath
                spikeId = pd.read_csv(pathSpike)
                spikeId = np.array(spikeId)
                spikeId = spikeId.flatten()
                spikeId = spikeId.tolist()
            else:
                pathSpike = os.path.join(l,r'SpikeIdx.csv')
                print('LOL')
                IspikeId = pd.read_csv(pathSpike)
                IspikeId = np.array(IspikeId)
                IspikeId = IspikeId.flatten()
                IspikeId = IspikeId.tolist()
                VolMask = VolMask.astype(bool)

                # 2. Create a mapping from Old Index -> New Index
                # This creates an array where every index contains the count of "True" frames before it
                new_mapping = np.cumsum(VolMask) - 1 

                # 3. Filter and Map
                # We keep the spike IF the mask was True at that spot...
                # ...AND we convert it to the new index using our mapping.
                spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
        corrDict = fr_corr_and_change_points_scipy(cal_trace=TraceC, fs_ca=calSR,spike_idx_v=spikeId, fs_v=500,fr_bin_s=fr_sig,fr_smooth_sigma_s=cal_sig,ca_smooth_sigma_s=0.07,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
        save_path = os.path.join(l, f"corrDict{N}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path, "wb") as f:
                pickle.dump(corrDict, f)
        epoch_corr = windowed_fr_calcium_correlation(corrDict['fr_on_ca'],corrDict['ca_used'],corrDict['t_ca'],ws_s = ws,step_s=w_step,sigma_fr_s=0.1,sigma_ca_s=0.1)
        
        
        save_path_win = os.path.join(l, f"window_corr{N}_windowlen_{epoch_corr['window_size_s']}_frSm{fr_sig}_calSm{cal_sig}.pkl")

        with open(save_path_win, "wb") as f:
            pickle.dump(epoch_corr, f)

        fig_corr, fig_overlay,fig_scatter,fig_sub,Mean_diff,ccWin = plot_window_corr_results_plotly_top(epoch_corr, r_thr=0.7)
        fig_corr_path_svg = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_corr_path_html = os.path.join(l,f'corr_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_diff_path_svg = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_diff_path_html = os.path.join(l,f'corr_to_diff_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        fig_overlay_path_svg = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html = os.path.join(l,f'best_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub, 1):
            fig_subplot_path_svg = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html = os.path.join(l,f'best_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg,fig_subplot_path_html)
        save_plotly_svg_html(fig_corr,fig_corr_path_svg,fig_corr_path_html)
        save_plotly_svg_html(fig_overlay,fig_overlay_path_svg,fig_overlay_path_html)
        save_plotly_svg_html(fig_scatter,fig_diff_path_svg,fig_diff_path_html)
        
        AllWinCor.append(epoch_corr["corr_list"])
        WinMeanDiffAbs,WinMeanDiff,WinMaxMinDiff = calculate_fr_diff(epoch_corr)
        AllWinMeanDiff.append(WinMeanDiff)
        AllWinMaxMinDiff.append(WinMaxMinDiff)
        AllWinMeanDiffAbs.append(WinMeanDiffAbs)
        AllMeanDiffMax.append(np.mean(WinMaxMinDiff))
        AllMeanDiffSeq.append(np.mean(WinMeanDiff))
        AllMeanDiffSeqAbs.append(np.mean(WinMeanDiffAbs))
        AllCorMeanWin.append(np.mean(ccWin))
        AllMeanFR.append(np.mean(corrDict['fr_on_ca']))


        fig_corr_l, fig_overlay_l,fig_scatter_l,fig_sub_l = plot_window_corr_results_plotly_bot(epoch_corr, r_thr=0.1)
        
        fig_overlay_path_svg_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg')
        fig_overlay_path_html_l = os.path.join(l,f'worse_corr_win_windowS_{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.html')
        for i, fig_s in enumerate(fig_sub_l, 1):
            fig_subplot_path_svg_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.svg')
            fig_subplot_path_html_l = os.path.join(l,f'worse_corr_subplot{ws}_stepS_{w_step}_page{i}_frSm{fr_sig}_calSm{cal_sig}.html')
            save_plotly_svg_html(fig_s,fig_subplot_path_svg_l,fig_subplot_path_html_l)
        
        save_plotly_svg_html(fig_overlay_l,fig_overlay_path_svg_l,fig_overlay_path_html_l)



        lag_res = lag_scan_pearson(
            x=corrDict['fr_on_ca'],
            y=corrDict['ca_used'],
            fs=calSR,
            max_lag_s=3.0,
            lag_step_s=1/30,
            min_overlap_s=1.0,
            mode="max_pos"
        )
        r_obs = corrDict["pearson_r"]
        r_opt = lag_res["best_r"]
        AllBcorr.append(r_opt)
        AllCor.append(r_obs)
        state_label = (
            'motor' if isinstance(N, str) and N.startswith('m') else
            'rest' if isinstance(N, str) and N.startswith('r') else
            bsPyr[i].lower()
        )
        corr_summary_rows.append({
            'link': l,
            'brain state': str(state_label),
            'optimal lag time': float(lag_res['best_lag_s']),
            'basic r score': float(r_obs),
            'optimal r score': float(r_opt),
        })
        calEvnDic = optionB_ca_event_windows_compare_fr(ca_dff=TraceC,fs_ca=calSR,voltage_trace=TraceV,fs_v=500,ca_smooth_sigma_s=0.07,spike_idx_v=spikeId,save_path=l)
        # 1) Event-triggered plot (uses ca_event_traces if you saved them; otherwise falls back to onsets)
        fig1 = plot_event_triggered_fr_and_ca(calEvnDic,l)

        # 2) Whole-trace plot (uses ONLY ca_onsets_used_idx from the dict; boxes end at threshold drop)
        fig2 = plot_whole_trace_voltage_calcium_with_events(voltage_trace=TraceV, VolXaX=VolAX,ca_dff=TraceC, CalXax=CalAX,spike_idx_v=spikeId,calciumeventsDic=calEvnDic,out_dir=l)
        #cur_SS_Vol,cur_SS_Cal,cal_SS_idx = get_Vol_Cal_SS(single_spikes_list,VolTrace,CalTrace,VolAX,CalAX,save_path=txtPss)
        fig3 = plot_fr_basic(TraceV,VolAX,spikeId,corrDict['ca_used'],CalAX,corrDict['fr_on_ca'],fr_change_idx=corrDict['fr_change_idx'],pearson_p=corrDict['pearson_p'],pearson_r=corrDict['pearson_r'],path = l)
        fig = add_highcorr_window_markers(fig3, epoch_corr, thr=0.5)

        # 3️⃣ Save the marked version separately
        out_base_marked = os.path.join(l, f"base_FR_full_trace_highcorr_ws_{epoch_corr['window_size_s']}_frSm{fr_sig}_calSm{cal_sig}")
        _save_plotly_fig(fig, out_base_marked, N)
        TracePathCal = os.path.join(l,'changepoint.csv')
        # observed zero-lag correlation
        r_obs = corrDict["pearson_r"]

        # optimal lag correlation (from lag_scan_pearson)
        AnstOpt_r.append(lag_res['best_r'])
        AnstOpt_lag.append(lag_res['best_lag_s'])
        win_corr_dict = fr_corr_and_windowed(
                cal_trace=TraceC, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )

            # Append to state-level lists
        AnstWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
        AnstWindow_p.append(win_corr_dict['window_p_list'])
out_csv = f"Z:\\Adam-Lab-Shared\\Data\\Michal_Rubin\\data summery\\2026\\SST_corralation_sUMMARY_frSm{fr_sig}_calSm{cal_sig}.csv"
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df_summary = pd.DataFrame(corr_summary_rows)
cols = ['link','brain state','optimal lag time','basic r score','optimal r score']
df_summary = df_summary[[c for c in cols if c in df_summary.columns]]
df_summary.to_csv(out_csv, index=False)
# print(f"Saved correlation summary: {out_csv} ({len(corr_summary_rows)} rows)")

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\num

⚠️ No usable events for FR plot (fr_trials is None/empty).
   n_events_total: 0
   n_events_used : 0


c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\num

In [13]:


def add_global_linear_fit_and_r(fig, AllWinMeanDiff, AllWinCor):
    # ---- collect all finite points across all cells ----
    xs, ys = [], []
    n_cells = min(len(AllWinMeanDiff), len(AllWinCor))
    for ci in range(n_cells):
        x = np.asarray(AllWinMeanDiff[ci], dtype=float).ravel()
        y = np.asarray(AllWinCor[ci], dtype=float).ravel()
        m = np.isfinite(x) & np.isfinite(y)
        if m.any():
            xs.append(x[m])
            ys.append(y[m])

    if len(xs) == 0:
        return fig, np.nan

    X = np.concatenate(xs)
    Y = np.concatenate(ys)

    if X.size < 2 or np.std(X) == 0 or np.std(Y) == 0:
        return fig, np.nan

    # ---- linear fit ----
    a, b = np.polyfit(X, Y, 1)

    # ---- Pearson r over all pooled points ----
    r = float(np.corrcoef(X, Y)[0, 1])

    # ---- add fit line ----
    x_line = np.array([np.min(X), np.max(X)], dtype=float)
    y_line = a * x_line + b

    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        name="Global linear fit",
        line=dict(width=3),
    ))

    # ---- annotation: bottom-right of plot area ----
    fig.add_annotation(
        xref="paper", yref="paper",
        x=0.99, y=0.02,          # right + low
        xanchor="right", yanchor="bottom",
        text=f"Global fit: y = {a:.3g}x + {b:.3g}<br>Pearson r = {r:.3f}<br>n = {X.size}",
        showarrow=False,
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=12),
        align="right",
    )

    return fig, r
def add_global_linear_fit_and_r_xy(fig, AllWinX, AllWinY, fit_name="Global linear fit"):
    # ---- collect all finite points across all cells ----
    xs, ys = [], []
    n_cells = min(len(AllWinX), len(AllWinY))
    for ci in range(n_cells):
        x = np.asarray(AllWinX[ci], dtype=float).ravel()
        y = np.asarray(AllWinY[ci], dtype=float).ravel()
        m = np.isfinite(x) & np.isfinite(y)
        if m.any():
            xs.append(x[m])
            ys.append(y[m])

    if len(xs) == 0:
        return fig, np.nan

    X = np.concatenate(xs)
    Y = np.concatenate(ys)

    if X.size < 2 or np.std(X) == 0 or np.std(Y) == 0:
        return fig, np.nan

    # ---- linear fit ----
    a, b = np.polyfit(X, Y, 1)

    # ---- Pearson r over all pooled points ----
    r = float(np.corrcoef(X, Y)[0, 1])

    # ---- add fit line ----
    x_line = np.array([np.min(X), np.max(X)], dtype=float)
    y_line = a * x_line + b

    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        name=fit_name,
        line=dict(width=3),
    ))

    # ---- annotation: bottom-right of plot area ----
    fig.add_annotation(
        xref="paper", yref="paper",
        x=0.99, y=0.02,
        xanchor="right", yanchor="bottom",
        text=f"Global fit: y = {a:.3g}x + {b:.3g}<br>Pearson r = {r:.3f}<br>n = {X.size}",
        showarrow=False,
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=12),
        align="right",
    )

    return fig, r


In [18]:
from pathlib import Path

OUT_DIR = Path(r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026")

def save_fig(fig, stem):
    fig_path_svg = OUT_DIR / f"{stem}.svg"
    fig_path_html = OUT_DIR / f"{stem}.html"
    save_plotly_svg_html(fig, str(fig_path_svg), str(fig_path_html))

# 1) MeanDiff
fig = plot_corr_vs_meanDiff_by_cell(AllWinMeanDiff, AllWinCor, AllPath=AllPath)
fig, r_all = add_global_linear_fit_and_r(fig, AllWinMeanDiff, AllWinCor)
save_fig(fig, f"corr_vs_meanDiff_by_cell_MeanDiff_ws{ws}_stepS{w_step}_frSm{fr_sig}_calSm{cal_sig}")

# 2) MeanDiffAbs
fig = plot_corr_vs_meanDiff_by_cell(AllWinMeanDiffAbs, AllWinCor, AllPath=AllPath)
fig, r_all = add_global_linear_fit_and_r(fig, AllWinMeanDiffAbs, AllWinCor)
save_fig(fig, f"corr_vs_meanDiff_by_cell_MeanDiffAbs_ws{ws}_stepS{w_step}_frSm{fr_sig}_calSm{cal_sig}")

# 3) MaxMinDiff
fig = plot_corr_vs_meanDiff_by_cell(AllWinMaxMinDiff, AllWinCor, AllPath=AllPath)
fig, r_all = add_global_linear_fit_and_r(fig, AllWinMaxMinDiff, AllWinCor)
save_fig(fig, f"corr_vs_meanDiff_by_cell_MaxMinDiff_ws{ws}_stepS{w_step}_frSm{fr_sig}_calSm{cal_sig}")


<>:4: DeprecationWarning:

invalid escape sequence '\A'

<>:5: DeprecationWarning:

invalid escape sequence '\A'

<>:10: DeprecationWarning:

invalid escape sequence '\A'

<>:11: DeprecationWarning:

invalid escape sequence '\A'

<>:16: DeprecationWarning:

invalid escape sequence '\A'

<>:17: DeprecationWarning:

invalid escape sequence '\A'

<>:4: DeprecationWarning:

invalid escape sequence '\A'

<>:5: DeprecationWarning:

invalid escape sequence '\A'

<>:10: DeprecationWarning:

invalid escape sequence '\A'

<>:11: DeprecationWarning:

invalid escape sequence '\A'

<>:16: DeprecationWarning:

invalid escape sequence '\A'

<>:17: DeprecationWarning:

invalid escape sequence '\A'

C:\Users\owner\AppData\Local\Temp\ipykernel_16544\495195530.py:4: DeprecationWarning:

invalid escape sequence '\A'

C:\Users\owner\AppData\Local\Temp\ipykernel_16544\495195530.py:5: DeprecationWarning:

invalid escape sequence '\A'

C:\Users\owner\AppData\Local\Temp\ipykernel_16544\495195530.py:10: Depreca

ValueError: operands could not be broadcast together with shapes (56,) (0,) 

In [14]:
from pathlib import Path
import plotly.graph_objects as go

OUT_DIR = Path(r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX")
OUT_DIR.mkdir(parents=True, exist_ok=True)
tag = f"ws{ws}_stepS{w_step}_frSm{fr_sig}_calSm{cal_sig}"
def make_scatter_and_save(x, y, title, stem):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode="markers",
        marker=dict(size=6, opacity=0.75),
        name="points",
        showlegend=False
    ))
    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis_title="Mean |ΔFR| within window",
        yaxis_title="Pearson r (FR vs calcium)",
    )

    svg_path = OUT_DIR / f"{stem}.svg"
    html_path = OUT_DIR / f"{stem}.html"
    save_plotly_svg_html(fig, str(svg_path), str(html_path))
    return fig



make_scatter_and_save(
    AllMeanFR, AllBcorr,
    title=f"optimal correlation vs mean FR of cell {tag}",
    stem=f"full_best_corr_vs_meanfr_{tag}"
)
make_scatter_and_save(
    AllMeanDiffMax, AllBcorr,
    title=f"optimal correlation vs mean FR difference (Max-Min in window) {tag}",
    stem=f"full_best_corr_vs_meanDiff_MaxMin_{tag}"
)

make_scatter_and_save(
    AllMeanDiffSeq, AllBcorr,
    title=f"optimal correlation vs mean FR difference (mean diff in window) {tag}",
    stem=f"full_best_corr_vs_meanDiff_{tag}"
)

make_scatter_and_save(
    AllMeanDiffSeqAbs, AllBcorr,
    title=f"optimal correlation vs mean FR difference (abs mean diff in window) {tag}",
    stem=f"full_best_corr_vs_meanDiff_AbsVal_{tag}"
)

# -------------------------
# B) Full correlation (AllCor)
# -------------------------
make_scatter_and_save(
    AllMeanFR, AllCor,
    title=f"optimal correlation vs mean FR of cell {tag}",
    stem=f"full_corr_vs_meanfr_{tag}"
)
make_scatter_and_save(
    AllMeanDiffMax, AllCor,
    title=f"correlation vs mean FR difference (Max-Min in window) {tag}",
    stem=f"full_corr_vs_meanDiff_MaxMin_{tag}"
)

make_scatter_and_save(
    AllMeanDiffSeq, AllCor,
    title=f"correlation vs mean FR difference (mean diff in window) {tag}",
    stem=f"full_corr_vs_meanDiff_{tag}"
)

make_scatter_and_save(
    AllMeanDiffSeqAbs, AllCor,
    title=f"correlation vs mean FR difference (abs mean diff in window) {tag}",
    stem=f"full_corr_vs_meanDiff_AbsVal_{tag}"
)

# -------------------------
# C) Window-mean correlation (AllCorMeanWin)
# -------------------------
make_scatter_and_save(
    AllMeanDiffMax, AllCorMeanWin,
    title=f"window mean correlation vs mean FR difference (Max-Min in window) {tag}",
    stem=f"winMean_corr_vs_meanDiff_MaxMin_{tag}"
)

make_scatter_and_save(
    AllMeanDiffSeq, AllCorMeanWin,
    title=f"window mean correlation vs mean FR difference (mean diff in window) {tag}",
    stem=f"winMean_corr_vs_meanDiff_{tag}"
)

make_scatter_and_save(
    AllMeanDiffSeqAbs, AllCorMeanWin,
    title=f"window mean correlation vs mean FR difference (abs mean diff in window) {tag}",
    stem=f"winMean_corr_vs_meanDiff_AbsVal_{tag}"
)


In [120]:



fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=AllMeanDiffMax,
    y=AllBcorr,
    mode="markers",
        
    marker=dict(size=6, opacity=0.75),
    
   
))

fig2.update_layout(
    title=f'optimal corrlation as function of mean fr differance (Max in window - min in window) {ws} stepS {w_step} frSm{fr_sig} calSm{cal_sig}',
    template="plotly_white",
    xaxis_title="Mean |ΔFR| within window",
    yaxis_title="Pearson r (FR vs calcium)",
    legend_title_text="Cell / trace",
)

fig_path_svg = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_best_corr_vs_meanDiff_MaxMin{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg"
fig_path_html = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_best_corr_vs_meanDiff_MaxMin{ws}_stepS_{w_step}__frSm{fr_sig}_calSm{cal_sig}.html"
save_plotly_svg_html(fig2, fig_path_svg, fig_path_html)



fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=AllMeanDiffSeq,
    y=AllBcorr,
    mode="markers",
        
    marker=dict(size=6, opacity=0.75),
    
   
))

fig3.update_layout(
    title=f'optimal corrlation as function of mean fr differance (mean differance in window) {ws} stepS {w_step} frSm{fr_sig} calSm{cal_sig}',
    template="plotly_white",
    xaxis_title="Mean |ΔFR| within window",
    yaxis_title="Pearson r (FR vs calcium)",
    legend_title_text="Cell / trace",
)

fig_path_svg = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_best_corr_vs_meanDiff{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg"
fig_path_html = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_best_corr_vs_meanDiff{ws}_stepS_{w_step}__frSm{fr_sig}_calSm{cal_sig}.html"
save_plotly_svg_html(fig3, fig_path_svg, fig_path_html)

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=AllMeanDiffSeqAbs,
    y=AllBcorr,
    mode="markers",
        
    marker=dict(size=6, opacity=0.75),
    
   
))

fig3.update_layout(
    title=f'optimal corrlation as function of mean fr differance (mean differance in absalute value in window) {ws} stepS {w_step} frSm{fr_sig} calSm{cal_sig}',
    template="plotly_white",
    xaxis_title="Mean |ΔFR| within window",
    yaxis_title="Pearson r (FR vs calcium)",
    legend_title_text="Cell / trace",
)

fig_path_svg = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_best_corr_vs_meanDiff_AbsVal{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg"
fig_path_html = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_best_corr_vs_meanDiff_AbsVal{ws}_stepS_{w_step}__frSm{fr_sig}_calSm{cal_sig}.html"
save_plotly_svg_html(fig3, fig_path_svg, fig_path_html)




fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=AllMeanDiffMax,
    y=AllCor,
    mode="markers",
        
    marker=dict(size=6, opacity=0.75),
    
   
))

fig2.update_layout(
    title=f'corrlation as function of mean fr differance (Max in window - min in window) {ws} stepS {w_step} frSm{fr_sig} calSm{cal_sig}',
    template="plotly_white",
    xaxis_title="Mean |ΔFR| within window",
    yaxis_title="Pearson r (FR vs calcium)",
    legend_title_text="Cell / trace",
)

fig_path_svg = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_corr_vs_meanDiff_MaxMin{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg"
fig_path_html = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_corr_vs_meanDiff_MaxMin{ws}_stepS_{w_step}__frSm{fr_sig}_calSm{cal_sig}.html"
save_plotly_svg_html(fig2, fig_path_svg, fig_path_html)



fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=AllMeanDiffSeq,
    y=AllCor,
    mode="markers",
        
    marker=dict(size=6, opacity=0.75),
    
   
))

fig3.update_layout(
    title=f'corrlation as function of mean fr differance (mean differance in window) {ws} stepS {w_step} frSm{fr_sig} calSm{cal_sig}',
    template="plotly_white",
    xaxis_title="Mean |ΔFR| within window",
    yaxis_title="Pearson r (FR vs calcium)",
    legend_title_text="Cell / trace",
)

fig_path_svg = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_corr_vs_meanDiff{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg"
fig_path_html = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_corr_vs_meanDiff{ws}_stepS_{w_step}__frSm{fr_sig}_calSm{cal_sig}.html"
save_plotly_svg_html(fig3, fig_path_svg, fig_path_html)

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=AllMeanDiffSeqAbs,
    y=AllCor,
    mode="markers",
        
    marker=dict(size=6, opacity=0.75),
    
   
))

fig3.update_layout(
    title=f'corrlation as function of mean fr differance (mean differance in absalute value in window) {ws} stepS {w_step} frSm{fr_sig} calSm{cal_sig}',
    template="plotly_white",
    xaxis_title="Mean |ΔFR| within window",
    yaxis_title="Pearson r (FR vs calcium)",
    legend_title_text="Cell / trace",
)

fig_path_svg = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_corr_vs_meanDiff_AbsVal{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg"
fig_path_html = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\full_corr_vs_meanDiff_AbsVal{ws}_stepS_{w_step}__frSm{fr_sig}_calSm{cal_sig}.html"
save_plotly_svg_html(fig3, fig_path_svg, fig_path_html)



fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=AllMeanDiffMax,
    y=AllCorMeanWin,
    mode="markers",
        
    marker=dict(size=6, opacity=0.75),
    
   
))

fig2.update_layout(
    title=f'window mean corrlation as function of mean fr differance (Max in window - min in window) {ws} stepS {w_step} frSm{fr_sig} calSm{cal_sig}',
    template="plotly_white",
    xaxis_title="Mean |ΔFR| within window",
    yaxis_title="Pearson r (FR vs calcium)",
    legend_title_text="Cell / trace",
)

fig_path_svg = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\winMean_corr_vs_meanDiff_MaxMin{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg"
fig_path_html = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\winMean_corr_vs_meanDiff_MaxMin{ws}_stepS_{w_step}__frSm{fr_sig}_calSm{cal_sig}.html"
save_plotly_svg_html(fig2, fig_path_svg, fig_path_html)



fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=AllMeanDiffSeq,
    y=AllCorMeanWin,
    mode="markers",
        
    marker=dict(size=6, opacity=0.75),
    
   
))

fig3.update_layout(
    title=f'window mean corrlation as function of mean fr differance (mean differance in window) {ws} stepS {w_step} frSm{fr_sig} calSm{cal_sig}',
    template="plotly_white",
    xaxis_title="Mean |ΔFR| within window",
    yaxis_title="Pearson r (FR vs calcium)",
    legend_title_text="Cell / trace",
)

fig_path_svg = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\winMean_corr_vs_meanDiff{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg"
fig_path_html = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\winMean_corr_vs_meanDiff{ws}_stepS_{w_step}__frSm{fr_sig}_calSm{cal_sig}.html"
save_plotly_svg_html(fig3, fig_path_svg, fig_path_html)

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=AllMeanDiffSeqAbs,
    y=AllCorMeanWin,
    mode="markers",
        
    marker=dict(size=6, opacity=0.75),
    
   
))

fig3.update_layout(
    title=f'window mean corrlation as function of mean fr differance (mean differance in absalute value in window) {ws} stepS {w_step} frSm{fr_sig} calSm{cal_sig}',
    template="plotly_white",
    xaxis_title="Mean |ΔFR| within window",
    yaxis_title="Pearson r (FR vs calcium)",
    legend_title_text="Cell / trace",
)

fig_path_svg = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\winMean_corr_vs_meanDiff_AbsVal{ws}_stepS_{w_step}_frSm{fr_sig}_calSm{cal_sig}.svg"
fig_path_html = f"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX\winMean_corr_vs_meanDiff_AbsVal{ws}_stepS_{w_step}__frSm{fr_sig}_calSm{cal_sig}.html"
save_plotly_svg_html(fig3, fig_path_svg, fig_path_html)

<>:21: DeprecationWarning:

invalid escape sequence '\A'

<>:22: DeprecationWarning:

invalid escape sequence '\A'

<>:47: DeprecationWarning:

invalid escape sequence '\A'

<>:48: DeprecationWarning:

invalid escape sequence '\A'

<>:71: DeprecationWarning:

invalid escape sequence '\A'

<>:72: DeprecationWarning:

invalid escape sequence '\A'

<>:98: DeprecationWarning:

invalid escape sequence '\A'

<>:99: DeprecationWarning:

invalid escape sequence '\A'

<>:124: DeprecationWarning:

invalid escape sequence '\A'

<>:125: DeprecationWarning:

invalid escape sequence '\A'

<>:148: DeprecationWarning:

invalid escape sequence '\A'

<>:149: DeprecationWarning:

invalid escape sequence '\A'

<>:174: DeprecationWarning:

invalid escape sequence '\A'

<>:175: DeprecationWarning:

invalid escape sequence '\A'

<>:200: DeprecationWarning:

invalid escape sequence '\A'

<>:201: DeprecationWarning:

invalid escape sequence '\A'

<>:224: DeprecationWarning:

invalid escape sequence '\A'

<>:22

OSError: [Errno 22] Invalid argument: 'Z:\\Adam-Lab-Shared\\Data\\Michal_Rubin\\data summery\x826\\SST\\paramsEX\x0cull_best_corr_vs_meanDiff_MaxMin10_stepS_2_frSm0.45_calSm0.45.svg'

In [ ]:

## pyramidal version

def _load_final_spikes(folder, name=None):
    candidates = []
    if name:
        candidates.append(os.path.join(folder, f"final_spikes_{name}.pkl"))
    candidates.append(os.path.join(folder, "final_spikes.pkl"))
    for p in candidates:
        if os.path.exists(p):
            with open(p, "rb") as f:
                d = pickle.load(f)
            sp = None
            if isinstance(d, dict):
                sp = d.get("spike_indices", None)
                if sp is None:
                    sp = d.get("vm_all_spikes", None)
                if sp is None:
                    sp = d.get("spikes", None)
            if sp is None:
                continue
            sp = np.asarray(sp, dtype=int).ravel().tolist()
            return sp, d, p
    return None, None, None


MotorCalTran = []
MotorFrTran = []
RestCalTran = []
RestFrTran = []
AwakeCalTran = []
AwakeFrTran = []
AnstCalTran = []
AnstFrTran = []
MotorPears = []
MotorP_val = []
RestPears = []
RestP_val = []
AwakePears = []
AwakeP_val = []
AnstPears = []
AnstP_val = []
MotorCalDiff = []
MotorFRDiff = []
RestCalDiff = []
RestFRDiff = []
AwakeCalDiff = []
AwakeFRDiff = []
AnstCalDiff = []
AnstFRDiff = []
MotorOpt_r = []
MotorOpt_lag = []
RestOpt_r = []
RestOpt_lag = []
AwakeOpt_r = []
AwakeOpt_lag = []
AnstOpt_r = []
AnstOpt_lag = []
MotorWindow_r = []
MotorWindow_p = []
RestWindow_r = []
RestWindow_p = []
AwakeWindow_r = []
AwakeWindow_p = []
AnstWindow_r = []
AnstWindow_p = []
calSR = 30
volSR =500
corr_summary_rows = []
for i,l in enumerate(pathPyr):
    l = pathPyr[i]
    calSR= AllCalSR[i]
    #print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
    trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
    vol_mask = np.array(pd.read_csv(TracePathVolM)).flatten().astype(bool)
    cal_mask = np.array(pd.read_csv(TracePathCalM)).flatten().astype(bool)
    # Apply masks
    VolTrace = trace_vol_full[vol_mask]
    CalTrace = trace_cal_full[cal_mask]
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(VolTrace,0)]
    VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
    CalAX = np.linspace(0, (len(CalTrace)/calSR), len(CalTrace))
    path =os.path.join(l,r'SpikeIdx.csv')
    IspikeId = pd.read_csv(path)
    IspikeId = np.array(IspikeId)
    IspikeId = IspikeId.flatten()
    IspikeId = IspikeId.tolist()
    VolMask = vol_mask.astype(bool)

    # 2. Create a mapping from Old Index -> New Index
    # This creates an array where every index contains the count of "True" frames before it
    new_mapping = np.cumsum(vol_mask) - 1 

    # 3. Filter and Map
    # We keep the spike IF the mask was True at that spot...
    # ...AND we convert it to the new index using our mapping.
    spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(vol_mask) and vol_mask[int(i)]]
    long_all_spikes = sorted(list(set(spikeId)))
    long_all_spikes = sorted(spikeId)
    if bsPyr[i].lower()=='motor':
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()
        #calMot,calRes = Split_cal(changeP,TraceV,TraceC,VolAX,CalAX,motor)
        MotorSinglePikeIDX = []
        RestSinglePikeIDX = []
        MotorSinglePikeCalIDX = []
        RestSinglePikeCalIDX = []
        MotorBurstSpikeIDX = []
        RestBurstSpikeIDX = []
        MotorComplexSpikeIDX = []
        RestComplexSpikeIDX = []
        RestFRpath =os.path.join(l,r'FiringRateRest.csv')
        MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        
        dfMot = pd.read_csv(MotorFRpath)
        MotorFR = dfMot['fr'].tolist()
        dfRest = pd.read_csv(RestFRpath)
        RestFR = dfRest['fr'].tolist()
        for k in range(len(MotorFR)):
            TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
            TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/calSR), len(trace_cal_full))
            N = f'm{k}'
            
            if len(trace_vol_full) < 13 * 500:
                #print(f"SKIP {l} r{k}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
            if spikeId is None:
                curr_path_pkl = os.path.join(l, f'spike_detection_refined_new{N}.pkl')
                if os.path.exists(curr_path_pkl):
                    with open(curr_path_pkl, 'rb') as f:
                        data = pickle.load(f)
                    spike_indices = data['vm_all_spikes']
                    spikeId = sorted(spike_indices)
                else:
                    continue
            else:
                spikeId = sorted(spikeId)
            txtPss = os.path.join(l,'SinglS_par.txt')
            corrDict = fr_corr_and_change_points_scipy(cal_trace=trace_cal_full, fs_ca=calSR,spike_idx_v=spikeId, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=0.0,ca_smooth_sigma_s=0.0,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
            lag_res = lag_scan_pearson(
                x=corrDict['fr_on_ca'],
                y=corrDict['ca_used'],
                fs=calSR,
                max_lag_s=3.0,
                lag_step_s=1/30,
                min_overlap_s=1.0,
                mode="max_pos"
            )
            r_obs = corrDict["pearson_r"]
            r_opt = lag_res["best_r"]
            state_label = (
                'motor' if isinstance(N, str) and N.startswith('m') else
                'rest' if isinstance(N, str) and N.startswith('r') else
                bsPyr[i].lower()
            )
            corr_summary_rows.append({
                'link': l,
                'brain state': str(state_label),
                'optimal lag time': float(lag_res['best_lag_s']),
                'basic r score': float(r_obs),
                'optimal r score': float(r_opt),
            })
            calEvnDic = optionB_ca_event_windows_compare_fr(ca_dff=trace_cal_full,fs_ca=calSR,voltage_trace=trace_vol_full,fs_v=500, ca_smooth_sigma_s=0.07, spike_idx_v=spikeId,save_path=l,name = f'pyr{N}')
            # 1) Event-triggered plot (uses ca_event_traces if you saved them; otherwise falls back to onsets)
            
            fig1 = plot_event_triggered_fr_and_ca(calEvnDic,l,name = f'pyr{N}')

            # 2) Whole-trace plot (uses ONLY ca_onsets_used_idx from the dict; boxes end at threshold drop)
            fig2 = plot_whole_trace_voltage_calcium_with_events(voltage_trace=trace_vol_full, VolXaX=VolAX,ca_dff=trace_cal_full, CalXax=CalAX,spike_idx_v=spikeId,calciumeventsDic=calEvnDic,out_dir=l,name = f'pyr{N}')
            fig3 = plot_fr_basic(trace_vol_full,VolAX,spikeId,corrDict['ca_used'],CalAX,corrDict['fr_on_ca'],fr_change_idx=corrDict['fr_change_idx'],pearson_p=corrDict['pearson_p'],pearson_r=corrDict['pearson_r'],path = l,name=f'pyr{N}')
            
            #cur_SS_Vol,cur_SS_Cal,cal_SS_idx = get_Vol_Cal_SS(single_spikes_list,VolTrace,CalTrace,VolAX,CalAX,save_path=txtPss)
            MotorCalTran.append(calEvnDic['cal_trials'])
            MotorFrTran.append(calEvnDic['fr_trials'])
            MotorPears.append(corrDict['pearson_r'])
            MotorP_val.append(corrDict['pearson_p'])
            MotorCalDiff.append(corrDict['diff_cal_T'])
            MotorFRDiff.append(corrDict['diff_FR_T'])
            RestOpt_r.append(lag_res['best_r'])
            RestOpt_lag.append(lag_res['best_lag_s'])
            win_corr_dict = fr_corr_and_windowed(
                cal_trace=trace_cal_full, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )
            # observed zero-lag correlation
            r_obs = corrDict["pearson_r"]

            # optimal lag correlation (from lag_scan_pearson)
            r_opt = lag_res["best_r"]
            state_label = (
                'motor' if isinstance(N, str) and N.startswith('m') else
                'rest' if isinstance(N, str) and N.startswith('r') else
                bsPyr[i].lower()
            )
            corr_summary_rows.append({
                'link': l,
                'brain state': str(state_label),
                'optimal lag time': float(lag_res['best_lag_s']),
                'basic r score': float(r_obs),
                'optimal r score': float(r_opt),
            })

            # permutation null distribution of r
            r_null = corrDict["pearson_r_null"]
            alpha = 0.05
            r_crit = np.quantile(np.abs(r_null), 1 - alpha)
            perm_title = (
                f"Permutation test: FR–Ca correlation | "
                f"{bsPyr[i]} | {os.path.basename(l)} | {N}"
            )

            fig_perm = plot_perm_histogram_plotly(
                r_null=r_null,
                r_obs=r_obs,
                r_opt=r_opt,
                r_crit=r_crit,
                title=perm_title
            )

            svg_path = os.path.join(l, f'PermTest_FR_Ca_{N}.svg')
            html_path = os.path.join(l, f'PermTest_FR_Ca_{N}.html')
            save_plotly_svg_html(fig_perm, svg_path, html_path)
            # Append to state-level lists
            RestWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
            RestWindow_p.append(win_corr_dict['window_p_list'])
    elif bsPyr[i].lower()=='awake':
        N = 'main'
        spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
        if spikeId is None:
            spikePath = os.path.join(l, 'spike_detection_refined_new.pkl')
            spikePath2 = os.path.join(l, 'spike_detection_refined_newr1.pkl')
            spike_indices = []
            if os.path.exists(spikePath) or os.path.exists(spikePath2):
                if os.path.exists(spikePath):
                    with open(spikePath, 'rb') as f:
                        spike_data = pickle.load(f)
                    spike_indices = spike_data['vm_all_spikes']
                else:
                    with open(spikePath2, 'rb') as f:
                        spike_data = pickle.load(f)
                    spike_indices = spike_data['vm_all_spikes']
            spikeId = spike_indices
        long_all_spikes = spikeId
        txtPss = os.path.join(l,'SinglS_par.txt')
        corrDict = fr_corr_and_change_points_scipy(cal_trace=CalTrace, fs_ca=calSR,spike_idx_v= long_all_spikes, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=0.0,ca_smooth_sigma_s=0.0,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
        lag_res = lag_scan_pearson(
            x=corrDict['fr_on_ca'],
            y=corrDict['ca_used'],
            fs=calSR,
            max_lag_s=3.0,
            lag_step_s=1/30,
            min_overlap_s=1.0,
            mode="max_pos"
        )
        r_obs = corrDict["pearson_r"]
        r_opt = lag_res["best_r"]
        state_label = (
            'motor' if isinstance(N, str) and N.startswith('m') else
            'rest' if isinstance(N, str) and N.startswith('r') else
            bsPyr[i].lower()
        )
        corr_summary_rows.append({
            'link': l,
            'brain state': str(state_label),
            'optimal lag time': float(lag_res['best_lag_s']),
            'basic r score': float(r_obs),
            'optimal r score': float(r_opt),
        })
        calEvnDic = optionB_ca_event_windows_compare_fr(ca_dff=CalTrace,fs_ca=calSR,voltage_trace=VolTrace,fs_v=500,ca_smooth_sigma_s=0.07,spike_idx_v=long_all_spikes,save_path=l,name='_pyr')
        # 1) Event-triggered plot (uses ca_event_traces if you saved them; otherwise falls back to onsets)
        fig1 = plot_event_triggered_fr_and_ca(calEvnDic,l,name='_pyr')

        # 2) Whole-trace plot (uses ONLY ca_onsets_used_idx from the dict; boxes end at threshold drop)
        fig2 = plot_whole_trace_voltage_calcium_with_events(voltage_trace=VolTrace, VolXaX=VolAX,ca_dff=CalTrace, CalXax=CalAX,spike_idx_v=long_all_spikes,calciumeventsDic=calEvnDic,out_dir=l,name='_pyr')
        #cur_SS_Vol,cur_SS_Cal,cal_SS_idx = get_Vol_Cal_SS(single_spikes_list,VolTrace,CalTrace,VolAX,CalAX,save_path=txtPss)
        fig3 = plot_fr_basic(VolTrace,VolAX,long_all_spikes,corrDict['ca_used'],CalAX,corrDict['fr_on_ca'],fr_change_idx=corrDict['fr_change_idx'],pearson_p=corrDict['pearson_p'],pearson_r=corrDict['pearson_r'],path = l,name='_pyr')
            
        AwakeCalTran.append(calEvnDic['cal_trials'])
        AwakeFrTran.append(calEvnDic['fr_trials'])
        AwakePears.append(corrDict['pearson_r'])
        AwakeP_val.append(corrDict['pearson_p'])
        AwakeCalDiff.append(corrDict['diff_cal_T'])
        AwakeFRDiff.append(corrDict['diff_FR_T'])
        AwakeOpt_r.append(lag_res['best_r'])
        AwakeOpt_lag.append(lag_res['best_lag_s'])
        win_corr_dict = fr_corr_and_windowed(
                cal_trace=CalTrace, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )
        # observed zero-lag correlation
        r_obs = corrDict["pearson_r"]

        # optimal lag correlation (from lag_scan_pearson)
        r_opt = lag_res["best_r"]
        state_label = (
            'motor' if isinstance(N, str) and N.startswith('m') else
            'rest' if isinstance(N, str) and N.startswith('r') else
            bsPyr[i].lower()
        )
        corr_summary_rows.append({
            'link': l,
            'brain state': str(state_label),
            'optimal lag time': float(lag_res['best_lag_s']),
            'basic r score': float(r_obs),
            'optimal r score': float(r_opt),
        })

        # permutation null distribution of r
        r_null = corrDict["pearson_r_null"]
        alpha = 0.05
        r_crit = np.quantile(np.abs(r_null), 1 - alpha)
        perm_title = (
            f"Permutation test: FR–Ca correlation | "
            f"{bsPyr[i]} | {os.path.basename(l)}"
        )

        fig_perm = plot_perm_histogram_plotly(
            r_null=r_null,
            r_obs=r_obs,
            r_opt=r_opt,
            r_crit=r_crit,
            title=perm_title
        )

        svg_path = os.path.join(l, f'PermTest_FR_Ca_Pyr.svg')
        html_path = os.path.join(l, f'PermTest_FR_Ca_Pyr.html')
        save_plotly_svg_html(fig_perm, svg_path, html_path)
            # Append to state-level lists
        AwakeWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
        AwakeWindow_p.append(win_corr_dict['window_p_list'])
    elif bsPyr[i].lower()=='anst':
        N = 'main'
        spikeId, _spk_payload, _spk_path = _load_final_spikes(l, name=N)
        if spikeId is None:
            spikePath = os.path.join(l, 'spike_detection_refined_new.pkl')
            spikePath2 = os.path.join(l, 'spike_detection_refined_newr1.pkl')
            spike_indices = []
            if os.path.exists(spikePath) or os.path.exists(spikePath2):
                if os.path.exists(spikePath):
                    with open(spikePath, 'rb') as f:
                        spike_data = pickle.load(f)
                    spike_indices = spike_data['vm_all_spikes']
                else:
                    with open(spikePath2, 'rb') as f:
                        spike_data = pickle.load(f)
                    spike_indices = spike_data['vm_all_spikes']
            spikeId = spike_indices
        long_all_spikes = spikeId
        corrDict = fr_corr_and_change_points_scipy(cal_trace=CalTrace, fs_ca=calSR,spike_idx_v=long_all_spikes, fs_v=500,fr_bin_s=0.1,fr_smooth_sigma_s=0.0,ca_smooth_sigma_s=0.0,change_method="mad",change_k=3.0,
                min_separation_s=0.3,
            )
        lag_res = lag_scan_pearson(
            x=corrDict['fr_on_ca'],
            y=corrDict['ca_used'],
            fs=calSR,
            max_lag_s=3.0,
            lag_step_s=1/30,
            min_overlap_s=1.0,
            mode="max_pos"
        )
        r_obs = corrDict["pearson_r"]
        r_opt = lag_res["best_r"]
        state_label = (
            'motor' if isinstance(N, str) and N.startswith('m') else
            'rest' if isinstance(N, str) and N.startswith('r') else
            bsPyr[i].lower()
        )
        corr_summary_rows.append({
            'link': l,
            'brain state': str(state_label),
            'optimal lag time': float(lag_res['best_lag_s']),
            'basic r score': float(r_obs),
            'optimal r score': float(r_opt),
        })
        calEvnDic = optionB_ca_event_windows_compare_fr(ca_dff=CalTrace,fs_ca=calSR,voltage_trace=VolTrace,fs_v=500,ca_smooth_sigma_s=0.07,spike_idx_v=long_all_spikes,save_path=l,name='_pyr')
        # 1) Event-triggered plot (uses ca_event_traces if you saved them; otherwise falls back to onsets)
        fig1 = plot_event_triggered_fr_and_ca(calEvnDic,l,name='_pyr')

        # 2) Whole-trace plot (uses ONLY ca_onsets_used_idx from the dict; boxes end at threshold drop)
        fig2 = plot_whole_trace_voltage_calcium_with_events(voltage_trace=VolTrace, VolXaX=VolAX,ca_dff=CalTrace, CalXax=CalAX,spike_idx_v=long_all_spikes,calciumeventsDic=calEvnDic,out_dir=l,name='_pyr')
        #cur_SS_Vol,cur_SS_Cal,cal_SS_idx = get_Vol_Cal_SS(single_spikes_list,VolTrace,CalTrace,VolAX,CalAX,save_path=txtPss)
        fig3 = plot_fr_basic(VolTrace,VolAX,long_all_spikes,corrDict['ca_used'],CalAX,corrDict['fr_on_ca'],fr_change_idx=corrDict['fr_change_idx'],pearson_p=corrDict['pearson_p'],pearson_r=corrDict['pearson_r'],path = l,name='_pyr')
            
        AnstCalTran.append(calEvnDic['cal_trials'])
        AnstFrTran.append(calEvnDic['fr_trials'])
        AnstPears.append(corrDict['pearson_r'])
        AnstP_val.append(corrDict['pearson_p'])
        AnstCalDiff.append(corrDict['diff_cal_T'])
        AnstFRDiff.append(corrDict['diff_FR_T'])
        AnstOpt_r.append(lag_res['best_r'])
        AnstOpt_lag.append(lag_res['best_lag_s'])
        win_corr_dict = fr_corr_and_windowed(
                cal_trace=CalTrace, fs_ca=calSR,
                spike_idx_v=spikeId, fs_v=500,
                window_dur_s=10.0,
                fr_bin_s=0.1,
                fr_smooth_sigma_s=0.0,
                ca_smooth_sigma_s=0.07,
            )
        # observed zero-lag correlation
        r_obs = corrDict["pearson_r"]

        # optimal lag correlation (from lag_scan_pearson)
        r_opt = lag_res["best_r"]
        state_label = (
            'motor' if isinstance(N, str) and N.startswith('m') else
            'rest' if isinstance(N, str) and N.startswith('r') else
            bsPyr[i].lower()
        )
        corr_summary_rows.append({
            'link': l,
            'brain state': str(state_label),
            'optimal lag time': float(lag_res['best_lag_s']),
            'basic r score': float(r_obs),
            'optimal r score': float(r_opt),
        })

        # permutation null distribution of r
        r_null = corrDict["pearson_r_null"]
        alpha = 0.05
        r_crit = np.quantile(np.abs(r_null), 1 - alpha)
        perm_title = (
            f"Permutation test: FR–Ca correlation | "
            f"{bsPyr[i]} | {os.path.basename(l)}"
        )

        fig_perm = plot_perm_histogram_plotly(
            r_null=r_null,
            r_obs=r_obs,
            r_opt=r_opt,
            r_crit=r_crit,
            title=perm_title
        )

        svg_path = os.path.join(l, f'PermTest_FR_Ca_Pyr.svg')
        html_path = os.path.join(l, f'PermTest_FR_Ca_Pyr.html')
        save_plotly_svg_html(fig_perm, svg_path, html_path)

            # Append to state-level lists
        AnstWindow_r.append(win_corr_dict['window_r_list'])  # for Motor state
        AnstWindow_p.append(win_corr_dict['window_p_list'])
out_csv = r"Z:\\Adam-Lab-Shared\\Data\\Michal_Rubin\\data summery\\2026\\SST_corralation_sUMMARY.csv"
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df_summary = pd.DataFrame(corr_summary_rows)
cols = ['link','brain state','optimal lag time','basic r score','optimal r score']
df_summary = df_summary[[c for c in cols if c in df_summary.columns]]
df_summary.to_csv(out_csv, index=False)
# print(f"Saved correlation summary: {out_csv} ({len(corr_summary_rows)} rows)")

In [64]:


def drop_empty_flatten_and_pad(L, pad_value=np.nan):
    """
    L: list where each element is either:
       - [] / None
       - 1D numeric list/array
       - nested list like [ [..], [..], [], [..] ]  (ragged)
       - OR a pandas row/Series with cells that are lists (like your screenshot)

    Returns:
      out: 2D array (n_traces, max_len) padded with pad_value
    """

    traces = []

    for rec in L:
        if rec is None:
            continue

        # If it's a pandas Series / row, turn into python list of cells
        if hasattr(rec, "to_list"):
            rec = rec.to_list()

        # Empty recording
        if isinstance(rec, (list, tuple, np.ndarray)) and len(rec) == 0:
            continue

        # Try: direct 1D numeric
        try:
            arr = np.asarray(rec, float).ravel()
            if arr.size > 0:
                traces.append(arr)
                continue
        except Exception:
            pass

        # Otherwise: nested -> concatenate numeric parts
        parts = []
        for cell in rec:
            if cell is None:
                continue
            # skip empty cells
            if isinstance(cell, (list, tuple, np.ndarray)) and len(cell) == 0:
                continue
            try:
                a = np.asarray(cell, float).ravel()
                if a.size > 0:
                    parts.append(a)
            except Exception:
                continue

        if len(parts) == 0:
            continue

        traces.append(np.concatenate(parts))

    if len(traces) == 0:
        return np.zeros((0, 0), float)

    max_len = max(t.size for t in traces)
    out = np.full((len(traces), max_len), pad_value, dtype=float)

    for i, t in enumerate(traces):
        out[i, :t.size] = t

    return out


In [ ]:
print(type(AnstCalDiff[0]), "len:", len(AnstCalDiff[0]) if hasattr(AnstCalDiff[0], "__len__") else "NA")
print(type(AnstCalDiff[0][0]) if len(AnstCalDiff[0]) else "empty")

In [72]:
params = corrDict['params']
AnstCalDiff_pad = drop_empty_flatten_and_pad(AnstCalDiff)
AnstFRDiff_pad  = drop_empty_flatten_and_pad(AnstFRDiff)

AwakeCalDiff_pad = drop_empty_flatten_and_pad(AwakeCalDiff)
AwakeFRDiff_pad  = drop_empty_flatten_and_pad(AwakeFRDiff)

MotorCalDiff_pad = drop_empty_flatten_and_pad(MotorCalDiff)
MotorFRDiff_pad  = drop_empty_flatten_and_pad(MotorFRDiff)

RestCalDiff_pad = drop_empty_flatten_and_pad(RestCalDiff)
RestFRDiff_pad  = drop_empty_flatten_and_pad(RestFRDiff)
save_dir  = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"
fig = plot_event_summary_4states(
    out_svg=os.path.join(save_dir, "FR_Ca_4states.svg"),
    out_html=os.path.join(save_dir, "FR_Ca_4states.html"),
    AnstCalTran=AnstCalTran, AnstFrTran=AnstFrTran,
    AwakeCalTran=AwakeCalTran, AwakeFrTran=AwakeFrTran,
    MotorCalTran=MotorCalTran, MotorFrTran=MotorFrTran,
    RestCalTran=RestCalTran, RestFrTran=RestFrTran,info_box=False
)


fig = plot_event_summary_4states(
    out_svg=os.path.join(save_dir, "FR_Ca_forFRchange_4states.svg"),
    out_html=os.path.join(save_dir, "FR_Ca_forFRchange_4states.html"),
    AnstCalTran=AnstCalDiff_pad, AnstFrTran=AnstFRDiff_pad,
    AwakeCalTran=AwakeCalDiff_pad, AwakeFrTran=AwakeFRDiff_pad,
    MotorCalTran=MotorCalDiff_pad, MotorFrTran=MotorFRDiff_pad,
    RestCalTran=RestCalDiff_pad, RestFrTran=RestFRDiff_pad,
    thr_k=params.get("change_k"),ca_smooth_sigma_s=params.get("ca_smooth_sigma_s"),fr_smooth_sigma_s=params.get("fr_smooth_sigma_s"),fr_bin_s=params.get("fr_bin_s")
)



In [67]:
# ---- helpers ----
def fisher_z(r):
    r = np.asarray(r, float)
    r = r[np.isfinite(r)]
    # avoid inf at ±1
    r = np.clip(r, -0.999999, 0.999999)
    return np.arctanh(r)

def sem(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size < 2:
        return np.nan
    return np.std(x, ddof=1) / np.sqrt(x.size)

def holm_bonferroni(pvals):
    """Return adjusted p-values (Holm)."""
    pvals = np.asarray(pvals, float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m, float)
    for k, i in enumerate(order):
        adj[i] = min((m - k) * pvals[i], 1.0)
    # make monotone non-decreasing in sorted order
    adj_sorted = adj[order]
    for k in range(1, m):
        adj_sorted[k] = max(adj_sorted[k], adj_sorted[k-1])
    adj[order] = adj_sorted
    return adj

In [68]:
save_dir  = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"
states = {
        "Anst":  np.asarray(AnstPears, float),
        "Awake": np.asarray(AwakePears, float),
        "Motor": np.asarray(MotorPears, float),
        "Rest":  np.asarray(RestPears, float),
    }

colors = {
        "Anst":  "blue",
        "Awake": "magenta",
        "Motor": "orange",
        "Rest":  "green",
    }
order = ["Anst", "Awake", "Motor", "Rest"]
mean_lines = []
z_groups = []
group_names = []

means = []
sems = []
for state in order:
    vals = states[state]
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue
    mu = float(np.nanmean(vals))
    se = sem(vals)  # your sem() function
    mean_lines.append(f"{state}: mean r = {mu:.3f} ± {se:.3f} (n={len(vals)})")
    sems.append(se)

fig = go.Figure()

for state, vals in states.items():
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue

    # --- Box plot ---
    fig.add_trace(
        go.Box(
            y=vals,
            name=state,
            marker_color=colors[state],
            boxmean=True,        # shows mean as dashed line
            showlegend=False,
            opacity=0.5
        )
    )

    # --- Individual points ---
    fig.add_trace(
        go.Scatter(
            x=[state] * len(vals),
            y=vals,
            mode="markers",
            marker=dict(
                color=colors[state],
                size=8,
                line=dict(color="black", width=0.8)
            ),
                    # horizontal jitter
            opacity=0.85,
            showlegend=False
        )
    )

# ---- Mean box ----
fig.add_annotation(
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    text="<br>".join(mean_lines),
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12)
)


# --- Layout ---
fig.update_layout(
    title='Pearson correlation for fr to cal for diffrent brain states',
    yaxis_title="Pearson r (FR vs Calcium)",
    xaxis_title="Brain state",
    template="simple_white",
    width=900,
    height=550,
)

# Optional: constrain r range
fig.update_yaxes(range=[-1, 1])
saveAs = os.path.join(save_dir, "r_fr_to_cal")
_save_plotly_fig(fig,saveAs)

Saved:
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\r_fr_to_cal_.html
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\r_fr_to_cal_.svg


In [69]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
save_dir  = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"
# ---- Put your paired data here (r and mean FR must match 1:1 per point) ----
states_r = {
    "Anst":  np.asarray(AnstPears, float),
    "Awake": np.asarray(AwakePears, float),
    "Motor": np.asarray(MotorPears, float),
    "Rest":  np.asarray(RestPears, float),
}

states_fr = {
    "Anst":  np.asarray(AnstFRMean, float),
    "Awake": np.asarray(AwakeFRMean, float),
    "Motor": np.asarray(MotorFRMean, float),
    "Rest":  np.asarray(RestFRMean, float),
}

colors = {"Anst":"blue", "Awake":"magenta", "Motor":"orange", "Rest":"green"}
order = ["Anst", "Awake", "Motor", "Rest"]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=order
)

pos = {"Anst": (1,1), "Awake": (1,2), "Motor": (2,1), "Rest": (2,2)}

for state in order:
    r = states_r[state]
    fr = states_fr[state]

    # keep only finite paired points
    m = np.isfinite(r) & np.isfinite(fr)
    r = r[m]
    fr = fr[m]
    if r.size == 0:
        continue

    row, col = pos[state]

    fig.add_trace(
        go.Scatter(
            x=fr,
            y=r,
            mode="markers",
            marker=dict(color=colors[state], size=8, line=dict(color="black", width=0.8)),
            showlegend=False
        ),
        row=row, col=col
    )

    # Optional: add a simple least-squares trend line
    if r.size >= 2:
        p = np.polyfit(fr, r, 1)
        xline = np.linspace(fr.min(), fr.max(), 50)
        yline = p[0]*xline + p[1]
        fig.add_trace(
            go.Scatter(
                x=xline, y=yline,
                mode="lines",
                line=dict(color=colors[state], width=2),
                showlegend=False
            ),
            row=row, col=col
        )

# shared layout
fig.update_layout(
    title="Pearson r vs mean FR (per brain state)",
    template="simple_white",
    width=1000,
    height=750,
)

fig.update_xaxes(title_text="Mean firing rate (Hz)")
fig.update_yaxes(title_text="Pearson r", range=[-1, 1])

saveAs = os.path.join(save_dir, "r_vs_meanFR_4states")
_save_plotly_fig(fig, saveAs)
fig.show()


Saved:
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\r_vs_meanFR_4states_.html
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\r_vs_meanFR_4states_.svg


In [ ]:

save_dir  = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"
# ---- Put your paired data here (r and mean FR must match 1:1 per point) ----
states_r = {
    "Anst":  np.asarray(AnstPears, float),
    "Awake": np.asarray(AwakePears, float),
    "Motor": np.asarray(MotorPears, float),
    "Rest":  np.asarray(RestPears, float),
}

states_fr = {
    "Anst":  np.asarray(AnstFRdiffMean, float),
    "Awake": np.asarray(AwakeFRdiffMean, float),
    "Motor": np.asarray(MotorFRdiffMean, float),
    "Rest":  np.asarray(RestFRdiffMean, float),
}

colors = {"Anst":"blue", "Awake":"magenta", "Motor":"orange", "Rest":"green"}
order = ["Anst", "Awake", "Motor", "Rest"]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=order
)

pos = {"Anst": (1,1), "Awake": (1,2), "Motor": (2,1), "Rest": (2,2)}

for state in order:
    r = states_r[state]
    fr = states_fr[state]

    # keep only finite paired points
    m = np.isfinite(r) & np.isfinite(fr)
    r = r[m]
    fr = fr[m]
    if r.size == 0:
        continue

    row, col = pos[state]

    fig.add_trace(
        go.Scatter(
            x=fr,
            y=r,
            mode="markers",
            marker=dict(color=colors[state], size=8, line=dict(color="black", width=0.8)),
            showlegend=False
        ),
        row=row, col=col
    )

    # Optional: add a simple least-squares trend line
    if r.size >= 2:
        p = np.polyfit(fr, r, 1)
        xline = np.linspace(fr.min(), fr.max(), 50)
        yline = p[0]*xline + p[1]
        fig.add_trace(
            go.Scatter(
                x=xline, y=yline,
                mode="lines",
                line=dict(color=colors[state], width=2),
                showlegend=False
            ),
            row=row, col=col
        )

# shared layout
fig.update_layout(
    title="Pearson r vs mean FR (per brain state)",
    template="simple_white",
    width=1000,
    height=750,
)

fig.update_xaxes(title_text="Mean firing rate diffrence (Hz)")
fig.update_yaxes(title_text="Pearson r", range=[-1, 1])

saveAs = os.path.join(save_dir, "r_vs_meanFRdiff_4states")
_save_plotly_fig(fig, saveAs)
fig.show()


Saved:
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\r_vs_meanFRdiff_4states_.html
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\r_vs_meanFRdiff_4states_.svg


In [71]:
states = {
        "Anst":  np.asarray(AnstOpt_r, float),
        "Awake": np.asarray(AwakeOpt_r, float),
        "Motor": np.asarray(MotorOpt_r, float),
        "Rest":  np.asarray(RestOpt_r, float),
    }

colors = {
        "Anst":  "blue",
        "Awake": "magenta",
        "Motor": "orange",
        "Rest":  "green",
    }
order = ["Anst", "Awake", "Motor", "Rest"]
mean_lines = []
z_groups = []
group_names = []

means = []
sems = []
for state in order:
    vals = states[state]
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue
    mu = float(np.nanmean(vals))
    se = sem(vals)  # your sem() function
    mean_lines.append(f"{state}: mean r = {mu:.3f} ± {se:.3f} (n={len(vals)})")
    sems.append(se)

fig = go.Figure()

for state, vals in states.items():
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue

    # --- Box plot ---
    fig.add_trace(
        go.Box(
            y=vals,
            name=state,
            marker_color=colors[state],
            boxmean=True,        # shows mean as dashed line
            showlegend=False,
            opacity=0.5
        )
    )

    # --- Individual points ---
    fig.add_trace(
        go.Scatter(
            x=[state] * len(vals),
            y=vals,
            mode="markers",
            marker=dict(
                color=colors[state],
                size=8,
                line=dict(color="black", width=0.8)
            ),
                    # horizontal jitter
            opacity=0.85,
            showlegend=False
        )
    )

# ---- Mean box ----
fig.add_annotation(
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    text="<br>".join(mean_lines),
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12)
)


# --- Layout ---
fig.update_layout(
    title='Pearson correlation for fr to cal for diffrent brain states',
    yaxis_title="Pearson r (FR vs Calcium)",
    xaxis_title="Brain state",
    template="simple_white",
    width=900,
    height=550,
)

# Optional: constrain r range
fig.update_yaxes(range=[-1, 1])
saveAs = os.path.join(save_dir, "r_optimal_lag_fr_to_cal")
_save_plotly_fig(fig,saveAs)

Saved:
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\r_optimal_lag_fr_to_cal_.html
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\r_optimal_lag_fr_to_cal_.svg


In [21]:
states = {
        "Anst":  np.asarray(AnstOpt_lag, float),
        "Awake": np.asarray(AwakeOpt_lag, float),
        "Motor": np.asarray(MotorOpt_lag, float),
        "Rest":  np.asarray(RestOpt_lag, float),
    }

colors = {
        "Anst":  "blue",
        "Awake": "magenta",
        "Motor": "orange",
        "Rest":  "green",
    }
order = ["Anst", "Awake", "Motor", "Rest"]
mean_lines = []
z_groups = []
group_names = []

means = []
sems = []
for state in order:
    vals = states[state]
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue
    mu = float(np.nanmean(vals))
    se = sem(vals)  # your sem() function
    mean_lines.append(f"{state}: mean r = {mu:.3f} ± {se:.3f} (n={len(vals)})")
    sems.append(se)

fig = go.Figure()

for state, vals in states.items():
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue

    # --- Box plot ---
    fig.add_trace(
        go.Box(
            y=vals,
            name=state,
            marker_color=colors[state],
            boxmean=True,        # shows mean as dashed line
            showlegend=False,
            opacity=0.5
        )
    )

    # --- Individual points ---
    fig.add_trace(
        go.Scatter(
            x=[state] * len(vals),
            y=vals,
            mode="markers",
            marker=dict(
                color=colors[state],
                size=8,
                line=dict(color="black", width=0.8)
            ),
                    # horizontal jitter
            opacity=0.85,
            showlegend=False
        )
    )

# ---- Mean box ----
fig.add_annotation(
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    text="<br>".join(mean_lines),
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12)
)


# --- Layout ---
fig.update_layout(
    title='Pearson correlation for fr to cal for diffrent brain states',
    yaxis_title="Pearson r (FR vs Calcium)",
    xaxis_title="Brain state",
    template="simple_white",
    width=900,
    height=550,
)

# Optional: constrain r range
fig.update_yaxes(range=[-1, 1])
saveAs = os.path.join(save_dir, "optimal_lag_fr_to_cal")
_save_plotly_fig(fig,saveAs)

Saved:
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\optimal_lag_fr_to_cal_.html
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\optimal_lag_fr_to_cal_.svg


In [22]:
# Flatten the window correlations (similar to your drop_empty_flatten_and_pad function)
AnstWindow_r_flat = np.concatenate([np.asarray(x, float) for x in AnstWindow_r if len(x) > 0])
AwakeWindow_r_flat = np.concatenate([np.asarray(x, float) for x in AwakeWindow_r if len(x) > 0])
MotorWindow_r_flat = np.concatenate([np.asarray(x, float) for x in MotorWindow_r if len(x) > 0])
RestWindow_r_flat = np.concatenate([np.asarray(x, float) for x in RestWindow_r if len(x) > 0])

# Now use in your plotting code
states = {
    "Anst":  AnstWindow_r_flat,
    "Awake": AwakeWindow_r_flat,
    "Motor": MotorWindow_r_flat,
    "Rest":  RestWindow_r_flat,
}

colors = {
        "Anst":  "blue",
        "Awake": "magenta",
        "Motor": "orange",
        "Rest":  "green",
    }
order = ["Anst", "Awake", "Motor", "Rest"]
mean_lines = []
z_groups = []
group_names = []

means = []
sems = []
for state in order:
    vals = states[state]
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue
    mu = float(np.nanmean(vals))
    se = sem(vals)  # your sem() function
    mean_lines.append(f"{state}: mean r = {mu:.3f} ± {se:.3f} (n={len(vals)})")
    sems.append(se)

fig = go.Figure()

for state, vals in states.items():
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue

    # --- Box plot ---
    fig.add_trace(
        go.Box(
            y=vals,
            name=state,
            marker_color=colors[state],
            boxmean=True,        # shows mean as dashed line
            showlegend=False,
            opacity=0.5
        )
    )

    # --- Individual points ---
    fig.add_trace(
        go.Scatter(
            x=[state] * len(vals),
            y=vals,
            mode="markers",
            marker=dict(
                color=colors[state],
                size=8,
                line=dict(color="black", width=0.8)
            ),
                    # horizontal jitter
            opacity=0.85,
            showlegend=False
        )
    )

# ---- Mean box ----
fig.add_annotation(
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    text="<br>".join(mean_lines),
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12)
)


# --- Layout ---
fig.update_layout(
    title='Pearson correlation for fr to cal for diffrent brain states',
    yaxis_title="Pearson r (FR vs Calcium)",
    xaxis_title="Brain state",
    template="simple_white",
    width=900,
    height=550,
)

# Optional: constrain r range
fig.update_yaxes(range=[-1, 1])
saveAs = os.path.join(save_dir, "cor_for10sec")
_save_plotly_fig(fig,saveAs)

Saved:
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\cor_for10sec_.html
  Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\cor_for10sec_.svg
